# 09 store

Question: does `golem-store` keep the schema that `okf/store.md` lists, bring a store of the version
before to it with every row as it was, answer every call of the plugins and refuse every call that
breaks a rule of the store, above all the accounts with an amount that is negative or zero, an
overdraw and an unknown account, and does its file hold every change after a close, a prefix of the
changes after a kill between changes and after a writer that fails, refuse every file that is no
store of this Golem, and answer the calls of the server thread far inside the budget of 1 ms.

`golem-devtool store` runs a JSON script of calls on a store and prints the answer of every step
as a line as soon as it ran, `store-describe` prints the schema of a file as SQLite reports it,
and `store-bench` times the calls of the server thread on a filled store. The notebook holds every
answer to the expectation written beside its step, so a section reads as a list of calls and what
each must answer. The players are made up UUIDs and every file lies in a temporary directory.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The wiki page whose schema the file must match, and the seed of the permissions, both repository
# relative.
wiki = "okf/store.md"
seed = "config/permissions.toml"

# The rounds of changes before the flush and in the burst that the kill cuts, and the limit of the
# file size, in KiB, at which the writer must fail.
flushed_rounds = 1000
burst_rounds = 20000
size_limit_kib = 300

# The readings of the permissions while a second thread changes them, and the checks while a second
# thread changes the store.
race_rounds = 20000
race_checks = 200

# The size of the store of the cost section, the calls of each kind it times, and the budget of a
# call of the server thread in microseconds, DEFAULT_BUDGET of golem-plugin.
bench_players = 5000
bench_repeat = 2000
budget_us = 1000

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo,
rustc, `rusqlite` and `libsqlite3-sys`, whose SQLite the next section holds to the page. `steps`
runs a script of pairs of a step and its expectation and prints one row per step: an expectation is
the answer itself, `ANY` for any answer, a `Fails` with the kind and the message of the error, or a
function that judges the answer.

In [ ]:
import hashlib
import json
import math
import os
import re
import resource
import shutil
import signal
import sqlite3
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-09-"))
ENVIRONMENT = dict(os.environ, **QUIET)

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"
PACKAGES = {package["name"]: package["version"] for package in metadata["packages"]}

ALEX = "a1e5a1e5-0000-4000-8000-000000000001"
SAM = "5a305a30-0000-4000-8000-000000000002"
KIM = "c1a0c1a0-0000-4000-8000-000000000003"
ESSIE = "e55e0000-0000-4000-8000-000000000004"
NOBODY = "00000000-0000-4000-8000-0000000000ff"
NAMES = {ALEX: "Alex", SAM: "Sam", KIM: "Kim", ESSIE: "Essie", NOBODY: "nobody"}
LARGEST = 2**63 - 1
COUNTS = {"steps": 0, "refusals": 0, "scripts": 0}


class Any:
    def __repr__(self):
        return "ANY"


ANY = Any()


class Fails:
    """An expected error: its kind and its message, or a pattern the message must match."""

    def __init__(self, kind, message=None, pattern=None):
        self.kind, self.message, self.pattern = kind, message, pattern

    def __repr__(self):
        return "Fails(%r, %r)" % (self.kind, self.message or self.pattern)


def refused(message):
    return Fails("refused", message)


def place(dimension="minecraft:overworld", x=0.5, y=64.0, z=0.5, yaw=0.0, pitch=0.0):
    return {"dimension": dimension, "x": x, "y": y, "z": z, "yaw": yaw, "pitch": pitch}


def shown(text):
    text = text if isinstance(text, str) else json.dumps(text)
    for uuid, name in NAMES.items():
        text = text.replace(uuid, name)
    return text.replace(str(SCRATCH), "<scratch>")


def script(steps, stdin=None, preexec=None):
    COUNTS["scripts"] += 1
    path = SCRATCH / ("script-%03d.json" % COUNTS["scripts"])
    path.write_text(json.dumps({"steps": steps}), encoding="utf-8")
    result = subprocess.run([str(DEVTOOL), "store", path.name], cwd=SCRATCH, env=ENVIRONMENT, input=stdin,
                            capture_output=True, text=True, timeout=1200, preexec_fn=preexec)
    assert result.returncode == 0, "store exited %d\n%s" % (result.returncode, result.stderr)
    answers = [json.loads(line) for line in result.stdout.splitlines()]
    assert [answer["step"] for answer in answers] == list(range(len(steps))), result.stdout[-2000:]
    return answers, result.stderr


def judge(expected, answer):
    if isinstance(expected, Fails):
        error = answer.get("error")
        if not error or error["kind"] != expected.kind:
            return False
        if expected.kind == "refused":
            COUNTS["refusals"] += 1
        if expected.pattern is not None:
            return re.fullmatch(expected.pattern, error["message"]) is not None
        return expected.message is None or error["message"] == expected.message
    if "ok" not in answer:
        return False
    if expected is ANY:
        return True
    if callable(expected):
        return bool(expected(answer["ok"]))
    return answer["ok"] == expected


def steps(pairs, preexec=None):
    answers, stderr = script([step for step, _ in pairs], preexec=preexec)
    wrong = []
    for (step, expected), answer in zip(pairs, answers):
        good = judge(expected, answer)
        COUNTS["steps"] += 1
        outcome = answer["ok"] if "ok" in answer else answer["error"]
        print("%-3s %-26s %s" % ("ok" if good else "BAD", step["op"], shown(outcome)[:150]))
        if not good:
            wrong.append("step %d %s: expected %s, got %s" % (answer["step"], step["op"], expected, shown(answer)))
    assert not wrong, "\n".join(wrong)
    return [answer.get("ok") for answer in answers], stderr


def devtool(*arguments):
    return subprocess.run([str(DEVTOOL), *map(str, arguments)], cwd=SCRATCH, env=ENVIRONMENT,
                          capture_output=True, text=True, timeout=1200)


def open_store(path="store.db"):
    return ({"op": "open", "path": path}, ANY)


CLOSE = ({"op": "close"}, ANY)

print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("rusqlite:", PACKAGES.get("rusqlite"), "libsqlite3-sys:", PACKAGES.get("libsqlite3-sys"))

## The schema

A new file becomes a store of the schema version that `okf/store.md` names, with the application id
`Golm`, in the journal mode WAL, and every table of the store is STRICT and has the columns, the
keys, the foreign keys with what a delete does and the unique places that the section The schema of
`okf/store.md` lists for it, no more and no fewer. A closed copy of the new store then takes rows
through the `sqlite3` of Python, past the cache and the checks of the store: the file itself must
refuse every row that breaks a rule of the page, a `CHECK`, a foreign key, a unique place, a column
that takes no NULL or the type of a STRICT column, and delete the nodes of a group with the group.
It must take what the page allows where a reader might expect a refusal: a warp whose last owner is
no player of the store, a legacy mail without a sender name and a toggle that is left to the plugin.
The store runs the SQLite that the page names, of the `libsqlite3-sys` and the `rusqlite` it names,
and the `Cargo.toml` of the workspace pins both crates exactly. The connection of the writer runs
with each setting of the table of the section Durability, as SQLite reports it back on that
connection, since each holds per connection and a kill, which keeps the cache of the system, would
not show a store that syncs less.

In [ ]:
PAGE = (P.root / wiki).read_text(encoding="utf-8")
named = re.search(r"SQLite is the amalgamation (\S+) that `libsqlite3-sys` (\S+) bundles, through `rusqlite`\s+(\S+),", PAGE)
assert named, "%s names no SQLite" % wiki
sqlite_named, sys_named, rusqlite_named = named.groups()
PINS = tomllib.loads((P.root / "Cargo.toml").read_text(encoding="utf-8"))["workspace"]["dependencies"]
assert PINS["libsqlite3-sys"]["version"] == "=" + sys_named, PINS["libsqlite3-sys"]
assert PINS["rusqlite"]["version"] == "=" + rusqlite_named, PINS["rusqlite"]
assert (PACKAGES.get("libsqlite3-sys"), PACKAGES.get("rusqlite")) == (sys_named, rusqlite_named), PACKAGES
answers, _ = steps([({"op": "sqlite_version"}, sqlite_named)])
print("SQLite %s of libsqlite3-sys %s through rusqlite %s, each as %s names it" % (*named.groups(), wiki))
SECTION = PAGE.split("\n## Durability\n", 1)[1].split("\n## ", 1)[0]
WORDS = {"FULL": 2, "ON": True, "OFF": False}
DURABILITY = {}
for line in SECTION.splitlines():
    if line.startswith("| `"):
        name, value = [cell.strip().strip("`") for cell in line.strip().strip("|").split("|")][:2]
        DURABILITY[name] = WORDS.get(value, int(value) if value.isdigit() else value.lower())
assert sorted(DURABILITY) == ["busy_timeout", "foreign_keys", "fullfsync", "journal_mode", "locking_mode",
                              "synchronous", "trusted_schema"], DURABILITY
answers, _ = steps([open_store(), ({"op": "durability"}, DURABILITY), CLOSE])
described = devtool("store-describe", "store.db")
assert described.returncode == 0, described.stderr
SCHEMA = json.loads(described.stdout)


def wiki_tables():
    text = (P.root / wiki).read_text(encoding="utf-8")
    section = text.split("\n## The schema\n", 1)[1].split("\n## ", 1)[0]
    tables = {}
    for block in section.split("\n### ")[1:]:
        name, _, body = block.partition("\n")
        rows = [line for line in body.splitlines() if line.startswith("| `")]
        columns, keys = [], []
        for row in rows:
            column, kind, null, key, references, _ = [cell.strip() for cell in row.strip().strip("|").split("|")]
            column = column.strip("`")
            columns.append({"name": column, "type": kind, "not_null": null == "no",
                            "primary_key": int(key) if key else 0})
            if references:
                reference, _, cascade = references.partition(", ")
                assert cascade in ("", "deleted with it"), references
                table, _, to = reference.strip("`").partition(".")
                keys.append({"from": column, "table": table, "to": to,
                             "on_delete": "CASCADE" if cascade else "NO ACTION"})
        unique = sorted(list(pair) for pair in re.findall(r"The pair of `(\w+)` and `(\w+)` is unique", body))
        tables[name.strip()] = {"columns": columns, "foreign_keys": keys, "unique": unique}
    return tables


WIKI = wiki_tables()
assert SCHEMA["application_id"] == 0x476F6C6D, SCHEMA["application_id"]
VERSION = int(re.search(r"and the schema version (\d+) in its user\s+version", PAGE).group(1))
assert SCHEMA["version"] == VERSION, "the file has the schema version %s, %s names %d" % (
    SCHEMA["version"], wiki, VERSION)
assert SCHEMA["journal_mode"] == "wal", SCHEMA["journal_mode"]
assert SCHEMA["indexes"] == ["mail_by_receiver", "transfers_by_source", "transfers_by_target"], SCHEMA["indexes"]
names = [table["name"] for table in SCHEMA["tables"]]
assert sorted(WIKI) == names, "the file has the tables %s, the wiki %s" % (names, sorted(WIKI))
for table in SCHEMA["tables"]:
    page = WIKI[table["name"]]
    assert table["strict"], "%s is not STRICT" % table["name"]
    assert table["rows"] == 0, table
    assert table["columns"] == page["columns"], "%s: the file has %s, the wiki %s" % (
        table["name"], table["columns"], page["columns"])
    order = lambda keys: sorted((key["from"], key["table"], key["to"], key["on_delete"]) for key in keys)
    assert order(table["foreign_keys"]) == order(page["foreign_keys"]), "%s: %s against %s" % (
        table["name"], table["foreign_keys"], page["foreign_keys"])
    assert table["unique"] == page["unique"], "%s: the file has the unique %s, the wiki %s" % (
        table["name"], table["unique"], page["unique"])
    key = sorted((column["primary_key"], column["name"]) for column in table["columns"] if column["primary_key"])
    references = ", ".join("%s to %s.%s%s" % (*reference[:3], " deleted with it" if reference[3] == "CASCADE" else "")
                           for reference in order(table["foreign_keys"])) or "none"
    unique = "".join(", unique %s" % " and ".join(columns) for columns in table["unique"])
    print("%-26s %2d columns, key %s, foreign keys %s%s" % (
        table["name"], len(table["columns"]), ", ".join(name for _, name in key), references, unique))

RAW = SCRATCH / "raw.db"
shutil.copyfile(SCRATCH / "store.db", RAW)
PLAYER, OTHER = "00000000-0000-4000-8000-0000000000a1", "00000000-0000-4000-8000-0000000000a2"
TWO_STATES = ["teleport_enabled", "teleport_auto", "accepting_pay", "baltop_exempt", "social_spy"]
THREE_STATES = ["confirm_pay", "reply_to_last_recipient"]
WARP = "INSERT INTO warps (name, dimension, x, y, z, yaw, pitch, last_owner) VALUES ('%s', 'd', 0, 0, 0, 0, 0, ?)"
BREAKS = [
    ("a UUID in upper case", "INSERT INTO players (uuid, name, first_join, last_join, last_seen) VALUES (?, 'X', 0, 0, 0)",
     (PLAYER.upper().replace("A1", "B1"),), "CHECK constraint failed"),
    ("half a logout location", "UPDATE players SET logout_dimension = 'minecraft:overworld' WHERE uuid = ?", (PLAYER,),
     "CHECK constraint failed"),
    ("an afk of 2", "UPDATE players SET afk = 2 WHERE uuid = ?", (PLAYER,), "CHECK constraint failed"),
    *[("a %s of 2" % column, "UPDATE players SET %s = 2 WHERE uuid = ?" % column, (PLAYER,),
       "CHECK constraint failed") for column in TWO_STATES + THREE_STATES],
    *[("a %s of no value" % column, "UPDATE players SET %s = NULL WHERE uuid = ?" % column, (PLAYER,),
       "NOT NULL constraint failed: players.%s" % column) for column in TWO_STATES + ["last_teleport"]],
    ("a last teleport below zero", "UPDATE players SET last_teleport = -1 WHERE uuid = ?", (PLAYER,),
     "CHECK constraint failed"),
    ("a past name of an unknown player", "INSERT INTO player_names (player, position, name) VALUES (?, 0, 'Old')",
     (OTHER,), "FOREIGN KEY constraint failed"),
    ("a past name at a place below zero",
     "INSERT INTO player_names (player, position, name) VALUES (?, -1, 'Old')", (PLAYER,),
     "CHECK constraint failed"),
    ("two past names in one place", "INSERT INTO player_names (player, position, name) VALUES (?, 0, 'Older')",
     (PLAYER,), "UNIQUE constraint failed: player_names.player, player_names.position"),
    ("a setting of an unknown player", "INSERT INTO player_settings (player, key, value) VALUES (?, 'k', 'v')",
     (OTHER,), "FOREIGN KEY constraint failed"),
    ("a setting of a player twice", "INSERT INTO player_settings (player, key, value) VALUES (?, 'k', 'w')",
     (PLAYER,), "UNIQUE constraint failed: player_settings.player, player_settings.key"),
    ("a last owner of a warp in upper case", WARP % "upper", (PLAYER.upper(),), "CHECK constraint failed"),
    ("a last owner of a warp that is no UUID", WARP % "short", ("Alex",), "CHECK constraint failed"),
    ("a balance below zero", "UPDATE accounts SET balance = -1 WHERE player = ?", (PLAYER,), "CHECK constraint failed"),
    ("a balance as text", "UPDATE accounts SET balance = 'lots' WHERE player = ?", (PLAYER,),
     "cannot store TEXT value in INTEGER column accounts.balance"),
    ("an amount of zero", "INSERT INTO transfers (at, source, target, amount, note) VALUES (0, NULL, ?, 0, 'x')",
     (PLAYER,), "CHECK constraint failed"),
    ("an entry with neither account", "INSERT INTO transfers (at, amount, note) VALUES (0, 5, 'x')", (),
     "CHECK constraint failed"),
    ("an entry from an account to itself",
     "INSERT INTO transfers (at, source, target, amount, note) VALUES (0, ?, ?, 5, 'x')", (PLAYER, PLAYER),
     "CHECK constraint failed"),
    ("an entry of an unknown account", "INSERT INTO transfers (at, source, amount, note) VALUES (0, ?, 5, 'x')",
     (OTHER,), "FOREIGN KEY constraint failed"),
    ("an ignore of oneself", "INSERT INTO ignores (player, ignored) VALUES (?, ?)", (PLAYER, PLAYER),
     "CHECK constraint failed"),
    ("a home of an unknown player",
     "INSERT INTO homes (player, name, dimension, x, y, z, yaw, pitch) VALUES (?, 'base', 'd', 0, 0, 0, 0, 0)",
     (OTHER,), "FOREIGN KEY constraint failed"),
    ("a mail with the number 0",
     "INSERT INTO mail (id, sender_name, receiver, text, sent_at) VALUES (0, 'Console', ?, 'hi', 0)", (PLAYER,),
     "CHECK constraint failed"),
    ("a mail that expires at zero",
     "INSERT INTO mail (sender_name, receiver, text, sent_at, expires_at) VALUES ('Console', ?, 'hi', 0, 0)",
     (PLAYER,), "CHECK constraint failed"),
    ("a legacy mail with a sender name",
     "INSERT INTO mail (sender_name, receiver, text, sent_at, legacy) VALUES ('Console', ?, 'hi', 0, 1)",
     (PLAYER,), "CHECK constraint failed"),
    ("a mail without a sender name that is no legacy mail",
     "INSERT INTO mail (sender_name, receiver, text, sent_at) VALUES ('', ?, 'hi', 0)", (PLAYER,),
     "CHECK constraint failed"),
    ("a legacy flag of 2",
     "INSERT INTO mail (sender_name, receiver, text, sent_at, legacy) VALUES ('', ?, 'hi', 0, 2)", (PLAYER,),
     "CHECK constraint failed"),
    ("a weight past 32 bits", "UPDATE permission_groups SET weight = 2147483648 WHERE name = 'g'", (),
     "CHECK constraint failed"),
    ("two nodes in one place", "INSERT INTO permission_group_nodes (group_name, position, node) VALUES ('g', 0, 'b.c')",
     (), "UNIQUE constraint failed: permission_group_nodes.group_name, permission_group_nodes.position"),
    ("a player of the permissions in upper case",
     "INSERT INTO permission_player_nodes (player, position, node) VALUES (?, 0, 'a.b')", (PLAYER.upper(),),
     "CHECK constraint failed"),
    ("the removal of a group a player names", "DELETE FROM permission_groups WHERE name = 'h'", (),
     "FOREIGN KEY constraint failed"),
]
raw = sqlite3.connect(RAW, isolation_level=None)
raw.execute("PRAGMA foreign_keys = ON")
for statement in (
    "INSERT INTO players (uuid, name, first_join, last_join, last_seen) VALUES ('%s', 'A', 0, 0, 0)" % PLAYER,
    "INSERT INTO accounts (player, balance) VALUES ('%s', 100)" % PLAYER,
    "INSERT INTO permission_groups (name, weight) VALUES ('g', 0), ('h', 0)",
    "INSERT INTO permission_group_parents (group_name, position, parent) VALUES ('g', 0, 'h')",
    "INSERT INTO permission_group_nodes (group_name, position, node) VALUES ('g', 0, 'a.b'), ('g', 1, 'c.d')",
    "INSERT INTO permission_player_groups (player, position, group_name) VALUES ('%s', 0, 'h')" % PLAYER,
    "INSERT INTO player_names (player, position, name) VALUES ('%s', 0, 'Old')" % PLAYER,
    "INSERT INTO player_settings (player, key, value) VALUES ('%s', 'k', 'v')" % PLAYER,
):
    raw.execute(statement)
for what, statement, values, error in BREAKS:
    try:
        raw.execute(statement, values)
    except sqlite3.IntegrityError as refusal:
        assert error in str(refusal), "the file refused %s with %r, not %r" % (what, str(refusal), error)
    else:
        raise AssertionError("the file took %s" % what)
TAKES = [
    ("a warp whose last owner is no player of the store", WARP % "far", (OTHER,)),
    ("a legacy mail without a sender name",
     "INSERT INTO mail (sender_name, receiver, text, sent_at, legacy) VALUES ('', ?, 'Sam: hi', 0, 1)", (PLAYER,)),
    ("a toggle that is left to the plugin",
     "UPDATE players SET confirm_pay = NULL, reply_to_last_recipient = NULL WHERE uuid = ?", (PLAYER,)),
]
for what, statement, values in TAKES:
    try:
        raw.execute(statement, values)
    except sqlite3.Error as refusal:
        raise AssertionError("the file refused %s: %s" % (what, refusal))
raw.execute("DELETE FROM permission_player_groups")
raw.execute("DELETE FROM permission_group_parents")
raw.execute("DELETE FROM permission_groups WHERE name = 'g'")
left = raw.execute("SELECT count(*) FROM permission_group_nodes").fetchone()[0]
assert left == 0, "the nodes of a removed group stayed: %d" % left
assert raw.execute("PRAGMA integrity_check").fetchall() == [("ok",)]
raw.close()
print("the file refused %d rows that break its rules, took %d that keep them and deleted the nodes of a removed "
      "group with it" % (len(BREAKS), len(TAKES)))

## The players

A first join creates a player with every toggle at its default, a later one sets the name, the last
join and the last seen time and names the name before. The calls of a quit, of `/back`, of AFK and
of a nickname change their field alone, a lookup by name takes the name of the last join in any
case, and `put_player` puts the whole record of the migration tool, each toggle as the record gives
it. A UUID is read in either case and written in lower case. A call for a player who never joined, a
bad name and a bad location are refused and change nothing.

In [ ]:
LOGOUT = place(x=100.25, y=70.0, z=-30.75, yaw=180.0, pitch=15.5)
BACK = place("minecraft:the_nether", x=-8.0, y=32.0, z=12.0, yaw=-90.0, pitch=0.0)
# The toggles of a player after their first join.
TOGGLES = {"teleport_enabled": True, "teleport_auto": False, "last_teleport": 0, "accepting_pay": True,
           "confirm_pay": None, "reply_to_last_recipient": None, "baltop_exempt": False, "social_spy": False}
ALEX_RECORD = {"uuid": ALEX, "name": "Alexis", "first_join": 1000, "last_join": 2000, "last_seen": 3000,
               "logout": LOGOUT, "back": BACK, "afk": True, "nickname": "Lexi", **TOGGLES}
SAM_RECORD = {"uuid": SAM, "name": "Sam", "first_join": 1100, "last_join": 1100, "last_seen": 1100,
              "logout": None, "back": None, "afk": False, "nickname": "lexi", **TOGGLES}
ESSIE_RECORD = {"uuid": ESSIE, "name": "Essie", "first_join": 500, "last_join": 600, "last_seen": 700,
                "logout": place("minecraft:the_end", x=1.0, y=2.0, z=3.0), "back": None, "afk": False,
                "nickname": None, "teleport_enabled": False, "teleport_auto": True, "last_teleport": 650,
                "accepting_pay": False, "confirm_pay": True, "reply_to_last_recipient": False,
                "baltop_exempt": True, "social_spy": True}
assert all(ESSIE_RECORD[name] != default for name, default in TOGGLES.items()), "a toggle of Essie is a default"
unknown = refused("the player %s is not in the store" % NOBODY)
answers, _ = steps([
    open_store(),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, {"first": True, "previous_name": None}),
    ({"op": "join", "uuid": SAM, "name": "Sam", "at": 1100}, {"first": True, "previous_name": None}),
    ({"op": "join", "uuid": KIM, "name": "Kim", "at": 1200}, {"first": True, "previous_name": None}),
    ({"op": "join", "uuid": ALEX.upper(), "name": "Alexis", "at": 2000}, {"first": False, "previous_name": "Alex"}),
    ({"op": "quit", "uuid": ALEX, "location": LOGOUT, "at": 3000}, None),
    ({"op": "set_back", "uuid": ALEX, "location": BACK}, None),
    ({"op": "set_afk", "uuid": ALEX, "afk": True}, None),
    ({"op": "set_nickname", "uuid": ALEX, "nickname": "Lexi"}, None),
    ({"op": "set_nickname", "uuid": SAM, "nickname": "lexi"}, None),
    ({"op": "set_back", "uuid": SAM, "location": BACK}, None),
    ({"op": "set_back", "uuid": SAM, "location": None}, None),
    ({"op": "set_last_seen", "uuid": KIM, "at": 4000}, None),
    ({"op": "put_player", **ESSIE_RECORD}, None),
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    ({"op": "player", "uuid": SAM}, SAM_RECORD),
    ({"op": "player", "uuid": ESSIE}, ESSIE_RECORD),
    ({"op": "player", "uuid": KIM}, lambda record: record["last_seen"] == 4000 and record["last_join"] == 1200),
    ({"op": "player", "uuid": NOBODY}, None),
    ({"op": "player_named", "name": "ALEXIS"}, ALEX_RECORD),
    ({"op": "player_named", "name": "Alex"}, None),
    ({"op": "players_nicknamed", "nickname": "LEXI"}, [ALEX_RECORD, SAM_RECORD]),
    ({"op": "player_count"}, 4),
    ({"op": "quit", "uuid": NOBODY, "location": LOGOUT, "at": 3000}, unknown),
    ({"op": "set_afk", "uuid": NOBODY, "afk": True}, unknown),
    ({"op": "join", "uuid": ALEX, "name": "", "at": 5000}, refused("the name of a player is empty")),
    ({"op": "join", "uuid": ALEX, "name": "x" * 65, "at": 5000},
     refused("the name of a player has 65 characters, more than 64")),
    ({"op": "join", "uuid": ALEX, "name": "Al\nex", "at": 5000},
     refused("the name of a player holds the control character '\\n'")),
    ({"op": "set_nickname", "uuid": ALEX, "nickname": "y" * 65},
     refused("a nickname has 65 characters, more than 64")),
    ({"op": "quit", "uuid": ALEX, "location": place("minecraft overworld"), "at": 5000},
     refused("the logout location names the dimension \"minecraft overworld\", which holds the character ' '")),
    ({"op": "quit", "uuid": ALEX, "location": place(""), "at": 5000},
     refused("the logout location names the dimension \"\", which is empty")),
    ({"op": "quit", "uuid": ALEX, "location": place(x="NaN"), "at": 5000},
     refused("the logout location has the x NaN, which is not a finite number")),
    ({"op": "set_back", "uuid": ALEX, "location": place(z="inf")},
     refused("the back location has the z inf, which is not a finite number")),
    ({"op": "player", "uuid": "steve"}, Fails("argument", '"uuid" "steve" is not a UUID')),
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    CLOSE,
])

## The toggles, the past names and the settings of a player

Each toggle of a player and the time of their last teleport is set alone and changes its field
alone, and the two toggles that may follow the default of the plugin take none as well. Once all
eight fields stand changed, `check` must find the file equal to the cache, so the change of each
column reaches the file for a player who stands. A join under another name makes the name before the
newest past name, with the time of the join, and keeps a name that stands there already, so a player
who returns to an old name has it twice. A past name finds no player. `put_past_names` puts a whole
list in place, the newest first, each name with its time or with none, as the migration tool carries
them over, and a later change of the name goes on top of it. A setting of a player is a text by key
that belongs to that player alone and is no setting of the server. A call for a player who never
joined, a last teleport below zero, a bad past name, a bad key and a bad value are refused and
change nothing.

In [ ]:
TOGGLED = {"teleport_enabled": False, "teleport_auto": True, "last_teleport": 7000, "accepting_pay": False,
           "confirm_pay": True, "reply_to_last_recipient": False, "baltop_exempt": True, "social_spy": True}
assert list(TOGGLED) == list(TOGGLES) and all(TOGGLED[name] != TOGGLES[name] for name in TOGGLES)
# Each toggle is set alone, and the record after it differs from the one before in that field alone.
SAM_TOGGLED, alone = SAM_RECORD, []
for name, value in TOGGLED.items():
    SAM_TOGGLED = dict(SAM_TOGGLED, **{name: value})
    alone += [({"op": "set_player_toggle", "uuid": SAM, "toggle": name, "value": value}, None),
              ({"op": "player", "uuid": SAM}, SAM_TOGGLED)]
SAM_TOGGLED = dict(SAM_TOGGLED, confirm_pay=None, reply_to_last_recipient=None)
CARRIED = [{"name": "Kimber", "changed_at": 5900}, {"name": "K1m", "changed_at": None}]
PAST_ALEX = [{"name": "Alex", "changed_at": 2000}]
PAST_KIM = [{"name": "Kimmy", "changed_at": 6400}, {"name": "Kimberly", "changed_at": 6300}, *CARRIED]
PLAYER_SETTINGS = {"last_kit": "6500", "note": "likes melons"}
below_zero = refused("the last teleport of a player is -1, below zero")
answers, _ = steps([
    open_store(),
    ({"op": "player", "uuid": SAM}, SAM_RECORD),
    *alone,
    ({"op": "check"}, lambda check: check["passed"]),
    ({"op": "set_player_toggle", "uuid": SAM, "toggle": "confirm_pay", "value": None}, None),
    ({"op": "set_player_toggle", "uuid": SAM, "toggle": "reply_to_last_recipient", "value": None}, None),
    ({"op": "player", "uuid": SAM}, SAM_TOGGLED),
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    ({"op": "set_player_toggle", "uuid": NOBODY, "toggle": "social_spy", "value": True}, unknown),
    ({"op": "set_player_toggle", "uuid": SAM, "toggle": "last_teleport", "value": -1}, below_zero),
    ({"op": "put_player", **dict(ESSIE_RECORD, last_teleport=-1)}, below_zero),
    ({"op": "player", "uuid": SAM}, SAM_TOGGLED),
    ({"op": "player", "uuid": ESSIE}, ESSIE_RECORD),
    ({"op": "past_names", "uuid": ALEX}, PAST_ALEX),
    ({"op": "past_names", "uuid": SAM}, []),
    ({"op": "past_names", "uuid": NOBODY}, []),
    ({"op": "join", "uuid": KIM, "name": "Kimberly", "at": 6000}, {"first": False, "previous_name": "Kim"}),
    ({"op": "join", "uuid": KIM, "name": "Kimberly", "at": 6100}, {"first": False, "previous_name": None}),
    ({"op": "join", "uuid": KIM, "name": "Kim", "at": 6200}, {"first": False, "previous_name": "Kimberly"}),
    ({"op": "join", "uuid": KIM, "name": "Kimberly", "at": 6250}, {"first": False, "previous_name": "Kim"}),
    ({"op": "past_names", "uuid": KIM}, [{"name": "Kim", "changed_at": 6250}, {"name": "Kimberly", "changed_at": 6200},
                                         {"name": "Kim", "changed_at": 6000}]),
    ({"op": "put_past_names", "uuid": KIM, "names": CARRIED}, None),
    ({"op": "past_names", "uuid": KIM}, CARRIED),
    ({"op": "join", "uuid": KIM, "name": "Kimmy", "at": 6300}, {"first": False, "previous_name": "Kimberly"}),
    ({"op": "join", "uuid": KIM, "name": "Kim", "at": 6400}, {"first": False, "previous_name": "Kimmy"}),
    ({"op": "past_names", "uuid": KIM}, PAST_KIM),
    ({"op": "player_named", "name": "KIM"}, lambda record: record["uuid"] == KIM and record["last_join"] == 6400),
    ({"op": "player_named", "name": "Kimberly"}, None),
    ({"op": "put_past_names", "uuid": SAM, "names": []}, None),
    ({"op": "past_names", "uuid": SAM}, []),
    ({"op": "put_past_names", "uuid": NOBODY, "names": CARRIED}, unknown),
    ({"op": "put_past_names", "uuid": KIM, "names": [{"name": "", "changed_at": None}]},
     refused("a past name is empty")),
    ({"op": "put_past_names", "uuid": KIM, "names": [*CARRIED, {"name": "k" * 65, "changed_at": None}]},
     refused("a past name has 65 characters, more than 64")),
    ({"op": "past_names", "uuid": KIM}, PAST_KIM),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "last_kit", "value": "6000"}, None),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "last_kit", "value": "6500"}, None),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "note", "value": "likes melons"}, None),
    ({"op": "set_player_setting", "uuid": SAM, "key": "note", "value": "likes pumpkins"}, None),
    ({"op": "player_setting", "uuid": ALEX, "key": "note"}, "likes melons"),
    ({"op": "player_setting", "uuid": SAM, "key": "note"}, "likes pumpkins"),
    ({"op": "player_setting", "uuid": KIM, "key": "note"}, None),
    ({"op": "player_setting", "uuid": NOBODY, "key": "note"}, None),
    ({"op": "player_settings", "uuid": ALEX}, PLAYER_SETTINGS),
    ({"op": "remove_player_setting", "uuid": SAM, "key": "note"}, True),
    ({"op": "remove_player_setting", "uuid": SAM, "key": "note"}, False),
    ({"op": "player_settings", "uuid": SAM}, {}),
    ({"op": "player_settings", "uuid": NOBODY}, {}),
    ({"op": "setting", "key": "note"}, None),
    ({"op": "set_player_setting", "uuid": NOBODY, "key": "note", "value": "x"}, unknown),
    ({"op": "remove_player_setting", "uuid": NOBODY, "key": "note"}, unknown),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "", "value": "x"}, refused("the key of a setting is empty")),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "k" * 65, "value": "x"},
     refused("the key of a setting has 65 characters, more than 64")),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "note", "value": "x" * 65537},
     refused("the value of a setting has 65537 bytes, more than 65536")),
    ({"op": "set_player_setting", "uuid": ALEX, "key": "note", "value": "a\u0000b"},
     refused("the value of a setting holds a NUL character")),
    ({"op": "player_settings", "uuid": ALEX}, PLAYER_SETTINGS),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])

## Homes, warps and spawns

A home belongs to a player and is found by its name byte for byte, so `Base` and `base` are two
homes. Setting a home, a warp or a spawn answers whether it replaced one, and removing answers
whether there was one. A warp carries the node a player needs for it, if any, and the player who set
it last as the call names them, who need not be a player of the store, so a call that names none
leaves the warp without one. The spawn of the group `default` is the one spawn of the server. A
coordinate or an angle of -0.0 in a home, a warp, a spawn, a logout or a back location comes back as
0.0, before a reopen as after it, since the file keeps a zero without its sign.

In [ ]:
BASE = place(x=10.5, y=64.0, z=-3.5, yaw=90.0)
BASE2 = place(x=11.5, y=65.0, z=-4.5, yaw=45.0, pitch=-10.0)
UPPER = place(x=0.0, y=80.0, z=0.0)
NETHER = place("minecraft:the_nether", x=1.0, y=40.0, z=1.0)
SHOP = place(x=200.0, y=70.0, z=200.0)
VAULT = place(x=-500.0, y=12.0, z=-500.0, yaw=270.0)
SPAWN = place(x=0.5, y=100.0, z=0.5)
ADMIN_SPAWN = place(x=50.5, y=100.0, z=50.5)
HOMES = {"base": BASE2, "nether": NETHER}
WARPS = [{"name": "vault", "location": VAULT, "permission": "essentials.warps.vault", "last_owner": NOBODY}]
SPAWNS = {"default": SPAWN}
answers, _ = steps([
    open_store(),
    ({"op": "set_home", "uuid": ALEX, "name": "base", "location": BASE}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "base", "location": BASE2}, True),
    ({"op": "set_home", "uuid": ALEX, "name": "Base", "location": UPPER}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "nether", "location": NETHER}, False),
    ({"op": "home", "uuid": ALEX, "name": "base"}, BASE2),
    ({"op": "homes", "uuid": ALEX}, {"Base": UPPER, **HOMES}),
    ({"op": "home_count", "uuid": ALEX}, 3),
    ({"op": "remove_home", "uuid": ALEX, "name": "Base"}, True),
    ({"op": "remove_home", "uuid": ALEX, "name": "Base"}, False),
    ({"op": "homes", "uuid": ALEX}, HOMES),
    ({"op": "home", "uuid": SAM, "name": "base"}, None),
    ({"op": "homes", "uuid": NOBODY}, {}),
    ({"op": "set_home", "uuid": NOBODY, "name": "base", "location": BASE}, unknown),
    ({"op": "remove_home", "uuid": NOBODY, "name": "base"}, unknown),
    ({"op": "set_home", "uuid": ALEX, "name": "", "location": BASE}, refused("the name of a home is empty")),
    ({"op": "set_home", "uuid": ALEX, "name": "h" * 65, "location": BASE},
     refused("the name of a home has 65 characters, more than 64")),
    ({"op": "set_home", "uuid": ALEX, "name": "base", "location": place(x="NaN")},
     refused("the location of a home has the x NaN, which is not a finite number")),
    ({"op": "set_warp", "name": "shop", "location": BASE, "permission": None, "owner": ALEX}, False),
    ({"op": "warp", "name": "shop"}, {"name": "shop", "location": BASE, "permission": None, "last_owner": ALEX}),
    ({"op": "set_warp", "name": "vault", "location": VAULT, "permission": "essentials.warps.vault",
      "owner": NOBODY}, False),
    ({"op": "set_warp", "name": "shop", "location": SHOP, "permission": None, "owner": None}, True),
    ({"op": "warp", "name": "vault"}, WARPS[0]),
    ({"op": "warps"}, [{"name": "shop", "location": SHOP, "permission": None, "last_owner": None}, *WARPS]),
    ({"op": "set_warp", "name": "bad", "location": BASE, "permission": "essentials warps"},
     refused("the permission of a warp \"essentials warps\" holds the character ' '")),
    ({"op": "set_warp", "name": "", "location": BASE}, refused("the name of a warp is empty")),
    ({"op": "remove_warp", "name": "shop"}, True),
    ({"op": "remove_warp", "name": "shop"}, False),
    ({"op": "warp", "name": "shop"}, None),
    ({"op": "set_spawn", "group": "default", "location": BASE}, False),
    ({"op": "set_spawn", "group": "admin", "location": ADMIN_SPAWN}, False),
    ({"op": "set_spawn", "group": "default", "location": SPAWN}, True),
    ({"op": "spawn", "group": "default"}, SPAWN),
    ({"op": "spawn", "group": "builder"}, None),
    ({"op": "spawns"}, {"admin": ADMIN_SPAWN, "default": SPAWN}),
    ({"op": "set_spawn", "group": "", "location": SPAWN}, refused("the group of a spawn is empty")),
    ({"op": "remove_spawn", "group": "admin"}, True),
    ({"op": "remove_spawn", "group": "admin"}, False),
    CLOSE,
])
NEGATIVE = place(x=-0.0, z=-0.0, yaw=-0.0, pitch=-0.0)


def unsigned(location):
    return location == place(x=0.0, z=0.0) and all(
        math.copysign(1.0, location[axis]) > 0 for axis in ("x", "z", "yaw", "pitch"))


ZEROS = [
    ({"op": "home", "uuid": ALEX, "name": "zero"}, unsigned),
    ({"op": "warp", "name": "zero"}, lambda warp: unsigned(warp["location"])),
    ({"op": "spawn", "group": "zero"}, unsigned),
    ({"op": "player", "uuid": ALEX}, lambda record: unsigned(record["logout"]) and unsigned(record["back"])),
    ({"op": "player", "uuid": SAM}, lambda record: unsigned(record["logout"]) and unsigned(record["back"])),
]
answers, _ = steps([
    open_store("zero.db"),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, ANY),
    ({"op": "set_home", "uuid": ALEX, "name": "zero", "location": NEGATIVE}, False),
    ({"op": "set_warp", "name": "zero", "location": NEGATIVE, "permission": None}, False),
    ({"op": "set_spawn", "group": "zero", "location": NEGATIVE}, False),
    ({"op": "quit", "uuid": ALEX, "location": NEGATIVE, "at": 2000}, None),
    ({"op": "set_back", "uuid": ALEX, "location": NEGATIVE}, None),
    ({"op": "put_player", **SAM_RECORD, "logout": NEGATIVE, "back": NEGATIVE}, None),
    *ZEROS,
    ({"op": "reopen"}, ANY),
    *ZEROS,
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])

## Mail and ignore lists

A mail comes from a player or from the console, which has no UUID, and gets the next number of the
store. Marking the mail of a player read answers how many were unread, and a mail can be removed
alone or with all others of its receiver. A mail may name the time at which it expires: the number
of unread mails then counts it only before that time, by the time that the call names, since the
store reads no clock, and the mail itself stays until a call removes it. An ignore list answers
whether a call changed it. A mail to or from a player who never joined, an empty or too long text,
an expiry that is not above zero, a player who ignores themselves and an unknown player on an ignore
list are refused.

In [ ]:
def mail(number, sender, sender_name, receiver, text, sent_at, read, expires_at=None, legacy=False):
    return {"id": number, "sender": sender, "sender_name": sender_name, "receiver": receiver, "text": text,
            "sent_at": sent_at, "read": read, "expires_at": expires_at, "legacy": legacy}


FIRST = mail(1, SAM, "Sam", ALEX, "hello Alex", 5000, False)
CONSOLE = mail(2, None, "Console", ALEX, "the server restarts at noon", 5100, False)
TO_SAM = mail(3, KIM, "Kim", SAM, "hi Sam", 5200, False)
TIMED = mail(4, ALEX, "Alexis", KIM, "the offer ends at noon", 5250, False, expires_at=6000)
WELCOME = mail(5, None, "Console", KIM, "welcome", 5260, False)
no_expiry = lambda at: refused("the expiry of a mail is %d, not above zero" % at)
answers, _ = steps([
    open_store(),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "hello Alex", "at": 5000}, 1),
    ({"op": "send_mail", "sender": None, "sender_name": "Console", "receiver": ALEX,
      "text": "the server restarts at noon", "at": 5100}, 2),
    ({"op": "send_mail", "sender": KIM, "sender_name": "Kim", "receiver": SAM, "text": "hi Sam", "at": 5200}, 3),
    ({"op": "unread_mail", "receiver": ALEX, "at": 5250}, 2),
    ({"op": "mail", "receiver": ALEX}, [FIRST, CONSOLE]),
    ({"op": "mark_mail_read", "receiver": ALEX}, 2),
    ({"op": "mark_mail_read", "receiver": ALEX}, 0),
    ({"op": "unread_mail", "receiver": ALEX, "at": 5250}, 0),
    ({"op": "mail", "receiver": ALEX}, [dict(FIRST, read=True), dict(CONSOLE, read=True)]),
    ({"op": "remove_mail", "receiver": ALEX, "id": 1}, True),
    ({"op": "remove_mail", "receiver": ALEX, "id": 1}, False),
    ({"op": "remove_mail", "receiver": ALEX, "id": 3}, False),
    ({"op": "mail", "receiver": ALEX}, [dict(CONSOLE, read=True)]),
    ({"op": "send_mail", "sender": ALEX, "sender_name": "Alexis", "receiver": KIM, "text": "the offer ends at noon",
      "at": 5250, "expires_at": 6000}, 4),
    ({"op": "send_mail", "sender": None, "sender_name": "Console", "receiver": KIM, "text": "welcome", "at": 5260,
      "expires_at": None}, 5),
    ({"op": "unread_mail", "receiver": KIM, "at": 5999}, 2),
    ({"op": "unread_mail", "receiver": KIM, "at": 6000}, 1),
    ({"op": "unread_mail", "receiver": KIM, "at": LARGEST}, 1),
    ({"op": "mail", "receiver": KIM}, [TIMED, WELCOME]),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "another", "at": 5300}, 6),
    ({"op": "clear_mail", "receiver": ALEX}, 2),
    ({"op": "clear_mail", "receiver": ALEX}, 0),
    ({"op": "mail", "receiver": ALEX}, []),
    ({"op": "mail", "receiver": SAM}, [TO_SAM]),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": NOBODY, "text": "hi", "at": 5400}, unknown),
    ({"op": "send_mail", "sender": NOBODY, "sender_name": "Ghost", "receiver": ALEX, "text": "boo", "at": 5400},
     unknown),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "", "at": 5400},
     refused("the text of a mail is empty")),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "x" * 65537, "at": 5400},
     refused("the text of a mail has 65537 bytes, more than 65536")),
    ({"op": "send_mail", "sender": SAM, "sender_name": "", "receiver": ALEX, "text": "hi", "at": 5400},
     refused("the name of the sender is empty")),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "hi", "at": 5400,
      "expires_at": 0}, no_expiry(0)),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "hi", "at": 5400,
      "expires_at": -1}, no_expiry(-1)),
    ({"op": "mail", "receiver": ALEX}, []),
    ({"op": "mark_mail_read", "receiver": NOBODY}, unknown),
    ({"op": "ignore", "player": ALEX, "other": SAM}, True),
    ({"op": "ignore", "player": ALEX, "other": SAM}, False),
    ({"op": "ignore", "player": ALEX, "other": KIM}, True),
    ({"op": "ignores", "player": ALEX, "other": SAM}, True),
    ({"op": "ignores", "player": SAM, "other": ALEX}, False),
    ({"op": "ignore_list", "player": ALEX}, [SAM, KIM]),
    ({"op": "unignore", "player": ALEX, "other": KIM}, True),
    ({"op": "unignore", "player": ALEX, "other": KIM}, False),
    ({"op": "ignore", "player": ALEX, "other": ALEX}, refused("the player %s cannot ignore themselves" % ALEX)),
    ({"op": "ignore", "player": ALEX, "other": NOBODY}, unknown),
    ({"op": "ignore", "player": NOBODY, "other": ALEX}, unknown),
    CLOSE,
])

## The accounts

An account opens with a starting balance, which the log records as a deposit when it is above
zero. A deposit adds money the server creates, a withdrawal takes money the server destroys, a
transfer moves money between two accounts in one step, and `set_balance` moves the difference, so
every change of a balance has its entry in the log. Every answer names the entry and the balances
after it, and `check` holds every balance to the sum of its log. That sum may pass the largest
integer of 64 bits on its way to a balance that fits: an account opened with the largest balance,
then a withdrawal and a deposit of one cent each, is a sound log that `check` must pass.

In [ ]:
def entry(number, at, source, target, amount, note):
    return {"id": number, "at": at, "source": source, "target": target, "amount": amount, "note": note}


def receipt(transfer, source=None, target=None):
    return {"transfer": transfer, "source_balance": source, "target_balance": target}


LEDGER_ALEX = [entry(7, 6500, ALEX, None, 500, "eco set"), entry(3, 6100, ALEX, SAM, 2500, "pay"),
               entry(1, 6000, None, ALEX, 10000, "starting balance")]
BALANCES = {ALEX: 7000, SAM: 2600, KIM: 1000}
answers, _ = steps([
    open_store(),
    ({"op": "create_account", "uuid": ALEX, "balance": 10000, "at": 6000, "note": "starting balance"},
     receipt(1, target=10000)),
    ({"op": "create_account", "uuid": SAM, "balance": 0, "at": 6000, "note": "starting balance"},
     receipt(None, target=0)),
    ({"op": "create_account", "uuid": KIM, "balance": 500, "at": 6000, "note": "starting balance"},
     receipt(2, target=500)),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 2500, "at": 6100, "note": "pay"},
     receipt(3, 7500, 2500)),
    ({"op": "deposit", "uuid": SAM, "amount": 100, "at": 6200, "note": "eco give"}, receipt(4, target=2600)),
    ({"op": "withdraw", "uuid": KIM, "amount": 200, "at": 6300, "note": "eco take"}, receipt(5, source=300)),
    ({"op": "set_balance", "uuid": KIM, "balance": 1000, "at": 6400, "note": "eco set"}, receipt(6, target=1000)),
    ({"op": "set_balance", "uuid": ALEX, "balance": 7000, "at": 6500, "note": "eco set"}, receipt(7, source=7000)),
    ({"op": "set_balance", "uuid": SAM, "balance": 2600, "at": 6600, "note": "eco set"}, receipt(None, target=2600)),
    ({"op": "balance", "uuid": ALEX}, 7000),
    ({"op": "balance", "uuid": SAM}, 2600),
    ({"op": "balance", "uuid": KIM}, 1000),
    ({"op": "balance", "uuid": ESSIE}, None),
    ({"op": "top_balances", "limit": 2}, [[ALEX, 7000], [SAM, 2600]]),
    ({"op": "top_balances", "limit": 10}, [[ALEX, 7000], [SAM, 2600], [KIM, 1000]]),
    ({"op": "account_count"}, 3),
    ({"op": "ledger", "account": ALEX, "limit": 10}, LEDGER_ALEX),
    ({"op": "ledger", "account": None, "limit": 3}, lambda entries: [e["id"] for e in entries] == [7, 6, 5]),
    ({"op": "check"}, lambda check: check["passed"] and check["rows"]["transfers"] == 7),
    CLOSE,
])
answers, _ = steps([
    open_store("largest.db"),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, ANY),
    ({"op": "create_account", "uuid": ALEX, "balance": LARGEST, "at": 1000, "note": "start"},
     receipt(1, target=LARGEST)),
    ({"op": "withdraw", "uuid": ALEX, "amount": 1, "at": 1100, "note": "eco take"}, receipt(2, source=LARGEST - 1)),
    ({"op": "deposit", "uuid": ALEX, "amount": 1, "at": 1200, "note": "eco give"}, receipt(3, target=LARGEST)),
    ({"op": "check"}, lambda check: check["passed"] and check["ledger_mismatches"] == []),
    CLOSE,
])

## The refusals of the accounts

Every change of an account that would break a rule is refused before anything moves: an amount
that is negative or zero, a balance below zero, an overdraw, an account that does not exist, a
transfer from an account to itself, a balance above the largest integer of 64 bits, a second
account of a player and a note that is too long. The checks run in the order the section
Refusals of `okf/store.md` names: the note, then the amount or the balance to set, then the same
account twice, then the accounts, the source first, then whether the source holds the amount and
the target can take it. So a change with a bad note is refused for its note whatever its amount,
and a transfer of zero cents to an unknown account is refused for its amount. After every refusal
the balances and the log are as they were.

In [ ]:
no_account = lambda uuid: refused("the player %s has no account" % uuid)
not_above = lambda amount: refused("the amount of %d cents is not above zero" % amount)
answers, _ = steps([
    open_store(),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": -5, "at": 7000, "note": "pay"}, not_above(-5)),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 0, "at": 7000, "note": "pay"}, not_above(0)),
    ({"op": "deposit", "uuid": ALEX, "amount": 0, "at": 7000}, not_above(0)),
    ({"op": "deposit", "uuid": ALEX, "amount": -1, "at": 7000}, not_above(-1)),
    ({"op": "withdraw", "uuid": ALEX, "amount": 0, "at": 7000}, not_above(0)),
    ({"op": "withdraw", "uuid": ALEX, "amount": -LARGEST - 1, "at": 7000}, not_above(-LARGEST - 1)),
    ({"op": "create_account", "uuid": ESSIE, "balance": -1, "at": 7000},
     refused("the balance of -1 cents is below zero")),
    ({"op": "set_balance", "uuid": ALEX, "balance": -1, "at": 7000},
     refused("the balance of -1 cents is below zero")),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 7001, "at": 7000, "note": "pay"},
     refused("the account of %s holds 7000 cents, less than the 7001 cents to take" % ALEX)),
    ({"op": "transfer", "source": SAM, "target": ALEX, "amount": LARGEST, "at": 7000, "note": "pay"},
     refused("the account of %s holds 2600 cents, less than the %d cents to take" % (SAM, LARGEST))),
    ({"op": "withdraw", "uuid": KIM, "amount": 1001, "at": 7000},
     refused("the account of %s holds 1000 cents, less than the 1001 cents to take" % KIM)),
    ({"op": "transfer", "source": ALEX, "target": ESSIE, "amount": 1, "at": 7000}, no_account(ESSIE)),
    ({"op": "transfer", "source": ESSIE, "target": ALEX, "amount": 1, "at": 7000}, no_account(ESSIE)),
    ({"op": "transfer", "source": ALEX, "target": NOBODY, "amount": 1, "at": 7000}, no_account(NOBODY)),
    ({"op": "transfer", "source": ALEX, "target": NOBODY, "amount": 0, "at": 7000}, not_above(0)),
    ({"op": "deposit", "uuid": NOBODY, "amount": 1, "at": 7000}, no_account(NOBODY)),
    ({"op": "withdraw", "uuid": NOBODY, "amount": 1, "at": 7000}, no_account(NOBODY)),
    ({"op": "set_balance", "uuid": NOBODY, "balance": 5, "at": 7000}, no_account(NOBODY)),
    ({"op": "transfer", "source": ALEX, "target": ALEX, "amount": 1, "at": 7000},
     refused("a transfer from the account of %s to itself moves nothing" % ALEX)),
    ({"op": "transfer", "source": NOBODY, "target": NOBODY, "amount": 5, "at": 7000},
     refused("a transfer from the account of %s to itself moves nothing" % NOBODY)),
    ({"op": "deposit", "uuid": SAM, "amount": LARGEST - 2599, "at": 7000},
     refused("the account of %s holds 2600 cents, and %d more would pass the largest balance of %d cents"
             % (SAM, LARGEST - 2599, LARGEST))),
    ({"op": "create_account", "uuid": ALEX, "balance": 0, "at": 7000},
     refused("the player %s has an account already" % ALEX)),
    ({"op": "create_account", "uuid": NOBODY, "balance": 0, "at": 7000}, unknown),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 1, "at": 7000, "note": "x" * 65537},
     refused("the note of a transfer has 65537 bytes, more than 65536")),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 1.5, "at": 7000},
     Fails("argument", '"amount" is not an integer')),
    ({"op": "create_account", "uuid": ESSIE, "balance": -1, "at": 7000, "note": "x" * 65537},
     refused("the note of a transfer has 65537 bytes, more than 65536")),
    ({"op": "set_balance", "uuid": ALEX, "balance": -1, "at": 7000, "note": "a\u0000b"},
     refused("the note of a transfer holds a NUL character")),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 0, "at": 7000, "note": "a\u0000b"},
     refused("the note of a transfer holds a NUL character")),
    ({"op": "deposit", "uuid": NOBODY, "amount": -1, "at": 7000, "note": "a\u0000b"},
     refused("the note of a transfer holds a NUL character")),
    ({"op": "balance", "uuid": ALEX}, BALANCES[ALEX]),
    ({"op": "balance", "uuid": SAM}, BALANCES[SAM]),
    ({"op": "balance", "uuid": KIM}, BALANCES[KIM]),
    ({"op": "ledger", "account": None, "limit": 100}, lambda entries: len(entries) == 7),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])

## The permissions

The seed `config/permissions.toml` goes into the store as a whole set, and the store answers it as
the same TOML, which `golem-devtool permissions` loads into the resolver of `golem-permissions`, so
a check of the set from the store decides every node of the seed as a check of the seed itself
does, with the same reason. A group and a player change alone and move the generation. The store
refuses what would break its rows: a group a list names but the store lacks, a node or a group
twice in a list, a name with white space, and the removal of a group that a group or a player
still names, and a player key of the TOML that is no UUID in lower case, as `golem-permissions`
refuses it. The cycles and the syntax of a node are the business of `golem-permissions`. A group,
the group of a spawn and a dimension are tokens of up to 256 bytes without white space, so a group
or the group of a spawn may have a longer name than the 64 characters of a home, while a home or a
warp may have a name with white space and the group of a spawn may not.

A resolver reloads the set through `permissions_with_generation`, which reads the generation and
the set under one lock. While a second thread changes the node of a player again and again, no
reading through it may pair a set with the generation of another set. The same readings through
`permissions` and then `permissions_generation`, with a yield of the thread between them where a
resolver does its work, must meet pairs that disagree, which shows that the changes reach the
readings in between. That is the order a resolver must not reload in, since such a pair keeps a
revoked node until the next change.

In [ ]:
SEED_TEXT = (P.root / seed).read_text(encoding="utf-8")
SEED = tomllib.loads(SEED_TEXT)
(SCRATCH / "seed.toml").write_text(SEED_TEXT, encoding="utf-8")
SAVED = {}


def keep(name, test=lambda value: True):
    def judge_and_keep(value):
        SAVED[name] = value
        return test(value)
    return judge_and_keep


def seeded(value):
    groups = {group["name"]: group for group in value["groups"]}
    return (value["generation"] == 1 and sorted(groups) == ["admin", "default"] and value["players"] == []
            and groups["default"]["nodes"] == SEED["groups"]["default"]["nodes"]
            and groups["admin"] == {"name": "admin", "weight": SEED["groups"]["admin"]["weight"],
                                    "parents": SEED["groups"]["admin"]["inherits"],
                                    "nodes": SEED["groups"]["admin"]["nodes"]})


NOT_A_UUID = "the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001"
ALEX_PERMISSIONS = {"uuid": ALEX, "groups": ["admin"], "nodes": ["-essentials.suicide"]}
BUILDER = {"name": "builder", "weight": 20, "parents": ["default"], "nodes": ["essentials.setwarp", "-essentials.suicide"]}
answers, _ = steps([
    open_store(),
    ({"op": "permissions"}, lambda value: value["groups"] == [] and value["players"] == [] and value["generation"] == 0),
    ({"op": "replace_permissions", "toml_file": "seed.toml"}, None),
    ({"op": "permissions"}, keep("seed", seeded)),
    ({"op": "put_player_permissions", **ALEX_PERMISSIONS}, None),
    ({"op": "player_permissions", "uuid": ALEX}, ALEX_PERMISSIONS),
    ({"op": "put_permission_group", "name": "builder", "weight": 10, "parents": ["default"],
      "nodes": ["essentials.setwarp"]}, False),
    ({"op": "put_permission_group", **BUILDER}, True),
    ({"op": "permission_group", "name": "builder"}, BUILDER),
    ({"op": "put_player_permissions", "uuid": SAM, "groups": ["builder"], "nodes": []}, None),
    ({"op": "permissions"}, lambda value: value["generation"] == 5 and len(value["players"]) == 2),
    ({"op": "put_permission_group", "name": "mod", "parents": ["moderator"]},
     refused('the group "moderator" is not in the permissions')),
    ({"op": "put_player_permissions", "uuid": KIM, "groups": ["vip"]}, refused('the group "vip" is not in the permissions')),
    ({"op": "put_permission_group", "name": "x", "nodes": ["essentials.home", "essentials.home"]},
     refused('the nodes of the group x name "essentials.home" twice')),
    ({"op": "put_player_permissions", "uuid": KIM, "groups": ["admin", "admin"]},
     refused('the groups of the player %s name "admin" twice' % KIM)),
    ({"op": "put_permission_group", "name": "bad group"},
     refused("the name of a group \"bad group\" holds the character ' '")),
    ({"op": "put_permission_group", "name": "y", "nodes": ["essentials home"]},
     refused("a node of a group \"essentials home\" holds the character ' '")),
    ({"op": "put_permission_group", "name": "builder", "weight": "10"}, Fails("argument", '"weight" is not an integer')),
    ({"op": "put_permission_group", "name": "builder", "weight": 1.5}, Fails("argument", '"weight" is not an integer')),
    ({"op": "put_permission_group", "name": "builder", "weight": 2**31},
     Fails("argument", '"weight" does not fit 32 bits')),
    ({"op": "permission_group", "name": "builder"}, BUILDER),
    ({"op": "remove_permission_group", "name": "default"},
     refused('the group "default" cannot be removed, since the group admin names it')),
    ({"op": "remove_permission_group", "name": "builder"},
     refused('the group "builder" cannot be removed, since the player %s names it' % SAM)),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[players.steve]\ngroups = []\n"},
     Fails("argument", "players.steve: " + NOT_A_UUID)),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[players.%s]\nnodes = [\"x.y\"]\n" % KIM.upper()},
     Fails("argument", "players.%s: %s" % (KIM.upper(), NOT_A_UUID))),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[players.\"a b\"]\n"},
     Fails("argument", 'players."a b": ' + NOT_A_UUID)),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[groups.admin]\ninherits = [\"moderator\"]\n"},
     refused('the group "moderator" is not in the permissions')),
    ({"op": "put_player_permissions", "uuid": SAM, "groups": [], "nodes": []}, None),
    ({"op": "player_permissions", "uuid": SAM}, None),
    ({"op": "remove_permission_group", "name": "builder"}, True),
    ({"op": "remove_permission_group", "name": "builder"}, False),
    ({"op": "remove_player_permissions", "uuid": SAM}, False),
    ({"op": "put_player_permissions", "uuid": KIM, "groups": [], "nodes": ["essentials.fly"]}, None),
    ({"op": "permissions"}, lambda value: value["generation"] == 8 and len(value["players"]) == 2),
    ({"op": "remove_player_permissions", "uuid": KIM}, True),
    ({"op": "player_permissions", "uuid": KIM}, None),
    ({"op": "permissions"}, keep("final", lambda value: value["generation"] == 9 and len(value["players"]) == 1)),
    CLOSE,
])
answers, _ = steps([
    open_store("race.db"),
    ({"op": "replace_permissions", "toml": "[groups.default]\nnodes = [\"essentials.home\"]\n"}, None),
    ({"op": "permissions_race", "rounds": race_rounds},
     keep("race", lambda race: race["disagree"] == 0 and race["two_calls_disagree"] > 0 and race["changes"] > 0
          and race["rounds"] == race_rounds)),
    ({"op": "player_permissions", "uuid": "7ace0000-0000-4000-8000-000000000001"}, None),
    ({"op": "permissions_generation"}, lambda generation: generation == SAVED["race"]["changes"] + 3),
    CLOSE,
])
print("%d readings while a second thread made %d changes: %d pairs of permissions_with_generation and %d pairs of two "
      "calls disagreed" % (race_rounds, SAVED["race"]["changes"], SAVED["race"]["disagree"],
                           SAVED["race"]["two_calls_disagree"]))
answers, _ = steps([
    open_store("tokens.db"),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, ANY),
    ({"op": "put_permission_group", "name": "g" * 65}, False),
    ({"op": "put_permission_group", "name": "g" * 256}, False),
    ({"op": "put_permission_group", "name": "g" * 257},
     refused('the name of a group "%s" has 257 bytes, more than 256' % ("g" * 257))),
    ({"op": "set_spawn", "group": "s" * 256, "location": place()}, False),
    ({"op": "set_spawn", "group": "s" * 257, "location": place()},
     refused('the group of a spawn "%s" has 257 bytes, more than 256' % ("s" * 257))),
    ({"op": "set_spawn", "group": "two words", "location": place()},
     refused("the group of a spawn \"two words\" holds the character ' '")),
    ({"op": "set_home", "uuid": ALEX, "name": "two words", "location": place()}, False),
    ({"op": "set_warp", "name": "two words", "location": place(), "permission": None}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "deep", "location": place("d" * 256)}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "deep", "location": place("d" * 257)},
     refused('the location of a home names the dimension "%s", which has 257 bytes, more than 256' % ("d" * 257))),
    CLOSE,
])

The set from the store and the seed must decide alike. The cell writes the TOML that the store
answered into a file and asks `golem-devtool permissions` for every node of the group default of
the seed and for admin nodes, for a player of the group default alone, once against the seed and
once against the TOML of the store. Alex is in the group admin in the store and denies
`essentials.suicide` of their own, which the resolver must read from the TOML of the store.

In [ ]:
STORE_TOML = SCRATCH / "store-permissions.toml"
STORE_TOML.write_text(SAVED["final"]["toml"], encoding="utf-8")
(SCRATCH / "store-seed.toml").write_text(SAVED["seed"]["toml"], encoding="utf-8")
NODES = SEED["groups"]["default"]["nodes"]
probe = [node.replace("*", "buy") for node in NODES] + ["essentials.setspawn", "essentials.eco", "golem.perm"]


def decisions(config, player):
    arguments = ["permissions", "--config", config, "--player", player]
    for node in probe:
        arguments += ["--node", node]
    result = devtool(*arguments)
    assert result.returncode == 0, "permissions refused %s: %s" % (config, result.stderr)
    return json.loads(result.stdout)["decisions"]


from_seed = decisions("seed.toml", KIM)
for name in ("store-seed.toml", "store-permissions.toml"):
    from_store = decisions(name, KIM)
    assert from_store == from_seed, "the set of %s decides otherwise:\n%s\n%s" % (name, from_store, from_seed)
assert sum(decision["value"] is True for decision in from_seed) == len(NODES), from_seed
# A decision also holds the answer of LuckPerms on Bukkit, which the cell leaves to 06_permissions.
alex = {decision["node"]: {key: decision[key] for key in ("node", "value", "reason")}
        for decision in decisions("store-permissions.toml", ALEX)}
assert alex["essentials.setspawn"] == {"node": "essentials.setspawn", "value": True, "reason": "Group admin grants essentials.*"}, alex
assert alex["essentials.suicide"] == {"node": "essentials.suicide", "value": False, "reason": "The player denies essentials.suicide"}, alex
assert alex["golem.perm"]["value"] is True, alex
print("%d nodes decided alike from the seed and from the store, and Alex holds the admin nodes but not essentials.suicide"
      % len(probe))

## The settings, and a closed store

A setting is a text by key. A key that is empty and a value that is too long or holds a NUL are
refused. After `close` every change, `flush` and `check` answer that the store is closed, a read
still answers from the cache, and a second close answers as the first. Two closes at once, while
the writer still has a burst of changes before it, must both wait until the writer wrote every
change and closed the file, and both answer that it did.

In [ ]:
SETTINGS = {"motd": "Welcome back"}
answers, _ = steps([
    open_store(),
    ({"op": "set_setting", "key": "motd", "value": "Welcome to SkyTekx"}, None),
    ({"op": "set_setting", "key": "rules", "value": "Be kind"}, None),
    ({"op": "set_setting", "key": "motd", "value": "Welcome back"}, None),
    ({"op": "setting", "key": "motd"}, "Welcome back"),
    ({"op": "setting", "key": "none"}, None),
    ({"op": "settings"}, {"motd": "Welcome back", "rules": "Be kind"}),
    ({"op": "remove_setting", "key": "rules"}, True),
    ({"op": "remove_setting", "key": "rules"}, False),
    ({"op": "set_setting", "key": "", "value": "x"}, refused("the key of a setting is empty")),
    ({"op": "set_setting", "key": "k", "value": "x" * 65537},
     refused("the value of a setting has 65537 bytes, more than 65536")),
    ({"op": "set_setting", "key": "k", "value": "a\u0000b"}, refused("the value of a setting holds a NUL character")),
    ({"op": "close"}, lambda stats: stats["closed"] and stats["pending"] == 0 and stats["failure"] is None
     and stats["written"] == stats["queued"]),
    ({"op": "set_setting", "key": "motd", "value": "late"}, Fails("closed", "the store is closed")),
    ({"op": "flush"}, Fails("closed", "the store is closed")),
    ({"op": "check"}, Fails("closed", "the store is closed")),
    ({"op": "setting", "key": "motd"}, "Welcome back"),
    ({"op": "close"}, lambda stats: stats["closed"] and stats["written"] == stats["queued"]),
])
VALUE = "{i} " + "x" * 500


def both_closed(value):
    sides = [value["first"], value["second"]]
    return all("ok" in side and side["ok"]["pending"] == 0 and side["ok"]["written"] == side["ok"]["queued"] == 30000
               and side["ok"]["failure"] is None for side in sides)


answers, _ = steps([
    open_store("closing.db"),
    ({"op": "repeat", "count": 30000, "steps": [{"op": "set_setting", "key": "burst", "value": VALUE}]}, ANY),
    ({"op": "stats"}, lambda stats: stats["pending"] > 0),
    ({"op": "close_concurrently"}, both_closed),
    ({"op": "set_setting", "key": "late", "value": "x"}, Fails("closed", "the store is closed")),
    ({"op": "close"}, lambda stats: stats["closed"] and stats["pending"] == 0),
    open_store("closing.db"),
    ({"op": "setting", "key": "burst"}, VALUE.replace("{i}", "30000")),
    CLOSE,
])

## A thread that changes faster than the writer

At most `MAX_PENDING` changes, the number the page names, wait for the writer. A script holds the
writer still for a second and queues 100000 settings as fast as it can, far faster than the writer
commits them, so changes find that many pending and wait for the next batch of the writer instead
of growing its queue. The most changes pending at once must stay at the bound and reach it, changes
must have waited, the log must name the wait once, and every change must reach the file.

In [ ]:
MAX_PENDING = int(re.search(r"\| `MAX_PENDING` \| (\d+),", PAGE).group(1))
FLOOD = 100000
answers, stderr = steps([
    open_store("flood.db"),
    ({"op": "pause_writer", "ms": 1000}, None),
    ({"op": "repeat", "count": FLOOD, "steps": [{"op": "set_setting", "key": "flood", "value": "{i} " + "x" * 100}]},
     ANY),
    ({"op": "stats"}, keep("flood", lambda stats: stats["largest_pending"] == MAX_PENDING and stats["waits"] > 0
                       and stats["pending"] <= MAX_PENDING)),
    ({"op": "close"}, lambda stats: stats["written"] == stats["queued"] == FLOOD and stats["pending"] == 0),
    open_store("flood.db"),
    ({"op": "setting", "key": "flood"}, "%d %s" % (FLOOD, "x" * 100)),
    CLOSE,
])
waited = "[golem] store: the writer of flood.db is %d changes behind, so a change waits until it takes the next batch" % (
    MAX_PENDING)
assert stderr.splitlines() == [waited], stderr
print("%d changes as fast as the script could make them: at most %d pending at once, %d changes waited for the writer"
      % (FLOOD, SAVED["flood"]["largest_pending"], SAVED["flood"]["waits"]))

## A reopen after a close

Every script above opened the same file and closed it, so each one already read what the one
before wrote. This cell reads everything once more, closes the store, opens it again and reads it
all a second time. Both readings must equal what the sections above left, `check` must find the
file sound, its accounts equal to their log and its rows equal to the cache, and the next mail
must take the number after the largest the store ever gave, though that mail was removed.

In [ ]:
READS = [
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    ({"op": "player", "uuid": ESSIE}, ESSIE_RECORD),
    ({"op": "player", "uuid": SAM}, SAM_TOGGLED),
    ({"op": "past_names", "uuid": ALEX}, PAST_ALEX),
    ({"op": "past_names", "uuid": KIM}, PAST_KIM),
    ({"op": "player_settings", "uuid": ALEX}, PLAYER_SETTINGS),
    ({"op": "player_count"}, 4),
    ({"op": "homes", "uuid": ALEX}, HOMES),
    ({"op": "warps"}, WARPS),
    ({"op": "spawns"}, SPAWNS),
    ({"op": "mail", "receiver": ALEX}, []),
    ({"op": "mail", "receiver": SAM}, [TO_SAM]),
    ({"op": "mail", "receiver": KIM}, [TIMED, WELCOME]),
    ({"op": "unread_mail", "receiver": KIM, "at": 6000}, 1),
    ({"op": "ignore_list", "player": ALEX}, [SAM]),
    ({"op": "balance", "uuid": ALEX}, BALANCES[ALEX]),
    ({"op": "balance", "uuid": SAM}, BALANCES[SAM]),
    ({"op": "top_balances", "limit": 10}, [[ALEX, 7000], [SAM, 2600], [KIM, 1000]]),
    ({"op": "ledger", "account": ALEX, "limit": 10}, LEDGER_ALEX),
    ({"op": "permissions"}, lambda value: value["toml"] == SAVED["final"]["toml"]),
    ({"op": "settings"}, SETTINGS),
    ({"op": "check"}, lambda check: check["passed"] and check["cache_differences"] == []),
]
answers, _ = steps([open_store(), *READS, ({"op": "reopen"}, ANY), *READS,
                    ({"op": "stats"}, lambda stats: stats["queued"] == 0),
                    ({"op": "send_mail", "sender": KIM, "sender_name": "Kim", "receiver": ALEX, "text": "after",
                      "at": 8000}, 7),
                    CLOSE])
before, after = answers[1:1 + len(READS)], answers[2 + len(READS):2 + 2 * len(READS)]
assert before[:-1] == after[:-1], "the reopen changed what the store answers"

## A check while the store changes

`check` compares the file with the cache as the call found it. A second thread changes a setting
and joins players every 100 microseconds while the script runs 200 checks, and no check may
report a difference, since the copy of the cache and the read of the file join the queue of the
writer together. Once the changes stop and reach the file, a quiet check passes as well.

In [ ]:
answers, _ = steps([
    open_store("check.db"),
    ({"op": "check_race", "checks": race_checks},
     lambda race: race["failed"] == 0 and race["parts"] == {} and race["changes"] > 0 and race["checks"] == race_checks),
    ({"op": "flush"}, None),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])

## A file that differs from the cache

`check` compares every table of the file with the cache, so a file that another program changed
behind the store does not pass. The script changes the file behind the cache through the hook of the
devtool, once in each part that the schema version 2 added, and `check` must name the part each
time: a setting of a player and a past name that the cache does not hold, and a toggle, the expiry
of a mail and the last owner of a warp that differ. The file stays sound in itself, so its integrity
and its foreign keys pass. The cache still answers what the store took, and the next open reads what
the file holds.

In [ ]:
def differs(*parts):
    return lambda check: (not check["passed"] and check["cache_differences"] == list(parts)
                          and check["integrity"] == ["ok"] and check["foreign_key_violations"] == 0)


behind = lambda sql: ({"op": "execute_behind", "sql": sql}, None)
answers, _ = steps([
    open_store("differs.db"),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, ANY),
    ({"op": "send_mail", "sender": None, "sender_name": "Console", "receiver": ALEX, "text": "hi", "at": 1100}, 1),
    ({"op": "set_warp", "name": "shop", "location": place(), "permission": None, "owner": ALEX}, False),
    ({"op": "check"}, lambda check: check["passed"]),
    behind("INSERT INTO player_settings (player, key, value) VALUES ('%s', 'behind', 'x')" % ALEX),
    ({"op": "check"}, differs("player settings")),
    behind("INSERT INTO player_names (player, position, name) VALUES ('%s', 0, 'Behind')" % ALEX),
    ({"op": "check"}, differs("past names", "player settings")),
    behind("UPDATE players SET social_spy = 1 WHERE uuid = '%s'" % ALEX),
    ({"op": "check"}, differs("players", "past names", "player settings")),
    behind("UPDATE mail SET expires_at = 2000 WHERE id = 1"),
    ({"op": "check"}, differs("players", "mail", "past names", "player settings")),
    behind("UPDATE warps SET last_owner = NULL WHERE name = 'shop'"),
    ({"op": "check"}, differs("players", "mail", "past names", "player settings", "warps")),
    ({"op": "player_settings", "uuid": ALEX}, {}),
    ({"op": "past_names", "uuid": ALEX}, []),
    ({"op": "reopen"}, ANY),
    ({"op": "player_settings", "uuid": ALEX}, {"behind": "x"}),
    ({"op": "past_names", "uuid": ALEX}, [{"name": "Behind", "changed_at": None}]),
    ({"op": "player", "uuid": ALEX}, lambda record: record["social_spy"] is True),
    ({"op": "mail", "receiver": ALEX}, lambda mails: [item["expires_at"] for item in mails] == [2000]),
    ({"op": "warp", "name": "shop"}, lambda warp: warp["last_owner"] is None),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])

## A store dropped without a close

A program that drops its last handle of the store without a close must still find every change
in the file: the store closes as its last handle goes and logs that it was not closed. The script
drops the store after a join and a setting, opens the file again and reads both back.

In [ ]:
answers, stderr = steps([
    open_store("drop.db"),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, {"first": True, "previous_name": None}),
    ({"op": "set_setting", "key": "dropped", "value": "before the drop"}, None),
    ({"op": "drop"}, None),
    ({"op": "setting", "key": "dropped"}, Fails("argument", "no store is open, the script needs an open step")),
    ({"op": "reopen"}, {"players": 1, "accounts": 0, "warps": 0, "settings": 1}),
    ({"op": "setting", "key": "dropped"}, "before the drop"),
    ({"op": "player", "uuid": ALEX}, lambda record: record["name"] == "Alex" and record["first_join"] == 1000),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])
dropped = "[golem] store: the store drop.db was not closed, so it closes as its last handle goes"
assert stderr.splitlines() == [dropped], stderr
print(dropped)

## A reopen after a kill between changes

A process opens a new store, makes two players with accounts, runs 1000 rounds of a setting and a
transfer of one cent, flushes, and then queues a burst of 20000 more rounds without a flush. The
notebook kills it with `SIGKILL` the moment the burst is queued, while the writer is still
committing it. The next open must find a sound file that holds every round before the flush and a
prefix of the burst: the setting of a round is in the file only if the transfer of every round
before it is, and the transfer of a round only with its setting. The money of the two accounts
must still sum to the start.

In [ ]:
A, B = ALEX, SAM
ROUND = lambda key, note: [{"op": "set_setting", "key": key, "value": "{i}"},
                           {"op": "transfer", "source": A, "target": B, "amount": 1, "at": 9000, "note": note}]
KILLED = [
    {"op": "open", "path": "kill.db"},
    {"op": "join", "uuid": A, "name": "Alex", "at": 9000},
    {"op": "join", "uuid": B, "name": "Sam", "at": 9000},
    {"op": "create_account", "uuid": A, "balance": 1000000, "at": 9000, "note": "start"},
    {"op": "create_account", "uuid": B, "balance": 0, "at": 9000},
    {"op": "repeat", "count": flushed_rounds, "steps": ROUND("flushed", "flushed {i}")},
    {"op": "flush"},
    {"op": "repeat", "count": burst_rounds, "steps": ROUND("burst", "burst {i}")},
    {"op": "wait"},
]
(SCRATCH / "kill.json").write_text(json.dumps({"steps": KILLED}), encoding="utf-8")
process = subprocess.Popen([str(DEVTOOL), "store", "kill.json"], cwd=SCRATCH, env=ENVIRONMENT, stdin=subprocess.PIPE,
                           stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
seen = []
for line in process.stdout:
    seen.append(json.loads(line))
    if seen[-1]["step"] == 7:
        break
os.kill(process.pid, signal.SIGKILL)
process.wait(timeout=60)
assert process.returncode == -signal.SIGKILL, process.returncode
assert all("ok" in answer for answer in seen), seen
print("the devtool queued the %d calls of the burst, %.1f microseconds a call with the reading of its steps, "
      "and got SIGKILL" % (seen[-1]["ok"]["calls"], seen[-1]["ok"]["nanos_per_call"] / 1000))
answers, _ = steps([
    open_store("kill.db"),
    ({"op": "check"}, lambda check: check["passed"]),
    ({"op": "setting", "key": "flushed"}, str(flushed_rounds)),
    ({"op": "setting", "key": "burst"}, ANY),
    ({"op": "balance", "uuid": A}, ANY),
    ({"op": "balance", "uuid": B}, ANY),
    ({"op": "ledger", "account": B, "limit": 1}, ANY),
    CLOSE,
])
_, _, _, burst, balance_a, balance_b, last, _ = answers
rounds = balance_b - flushed_rounds
settings_in = int(burst) if burst is not None else 0
assert balance_a + balance_b == 1000000, "the accounts hold %d cents, not 1000000" % (balance_a + balance_b)
assert 0 <= rounds < burst_rounds, "the file holds %d rounds of the burst, so the kill came after the writer" % rounds
assert rounds in (settings_in, settings_in - 1), "the file holds %d transfers but the setting of round %d" % (rounds, settings_in)
note = "burst %d" % rounds if rounds else "flushed %d" % flushed_rounds
assert last[0]["note"] == note and last[0]["id"] == 1 + flushed_rounds + rounds, last
print("the file holds the %d rounds before the flush and %d of the %d rounds of the burst, the setting of round %d,"
      " and %d cents in both accounts" % (flushed_rounds, rounds, burst_rounds, settings_in, balance_a + balance_b))

## A second process

While a process holds a store, a second process can neither read nor open it, so the cache of the
first stays the truth of the file. The first process opens a store and waits on its standard
input. `store-describe` and the open of a second script must answer that another process holds the
file. Once the standard input of the first ends, it closes the store, and the file opens again. A
second open of the file in the same process, under any path to it, is refused in its own words,
and once the store is closed the process opens the file again.

In [ ]:
HOLDER = SCRATCH / "hold.json"
HOLDER.write_text(json.dumps({"steps": [{"op": "open", "path": "kill.db"}, {"op": "wait"}]}), encoding="utf-8")
holder = subprocess.Popen([str(DEVTOOL), "store", HOLDER.name], cwd=SCRATCH, env=ENVIRONMENT, stdin=subprocess.PIPE,
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
assert json.loads(holder.stdout.readline())["op"] == "open"
assert json.loads(holder.stdout.readline())["op"] == "wait"
held = "cannot open the store kill.db: another process holds the file, so it is open in a second Golem or a tool"
described = devtool("store-describe", "kill.db")
assert described.returncode == 1 and described.stderr == "golem-devtool: store-describe: %s\n" % held, described
answers, _ = steps([({"op": "open", "path": "kill.db"}, Fails("open", held))])
holder.stdin.close()
assert holder.wait(timeout=60) == 0, holder.stderr.read()
assert devtool("store-describe", "kill.db").returncode == 0
twice = "the store is open in this process already"
answers, _ = steps([
    open_store("kill.db"),
    ({"op": "open_beside", "path": "kill.db"}, Fails("open", "cannot open the store kill.db: " + twice)),
    ({"op": "open_beside", "path": "./kill.db"}, Fails("open", "cannot open the store ./kill.db: " + twice)),
    CLOSE,
    ({"op": "open_beside", "path": "kill.db"}, None),
])

## A reader that goes away

`store` and `store-describe` write their answers on the standard output. When the reader of that
pipe is gone, each must stop with the exit code 1 and one line on the standard error instead of a
panic, and `store` must still close its store, so the steps that ran reach the file. The cell
gives `store-describe` a pipe whose reader is closed, and closes the reader of a script while the
script waits, so its next answer finds no reader.

In [ ]:
read_end, write_end = os.pipe()
os.close(read_end)
gone = subprocess.run([str(DEVTOOL), "store-describe", "kill.db"], cwd=SCRATCH, env=ENVIRONMENT, stdout=write_end,
                      stderr=subprocess.PIPE, text=True, timeout=600)
os.close(write_end)
print("store-describe:", gone.returncode, gone.stderr.strip())
assert gone.returncode == 1 and "panicked" not in gone.stderr, gone.stderr
assert re.fullmatch(r"golem-devtool: store-describe: the standard output is closed: .+\n", gone.stderr), gone.stderr
(SCRATCH / "gone.json").write_text(json.dumps({"steps": [
    {"op": "open", "path": "gone.db"},
    {"op": "set_setting", "key": "before", "value": "the wait"},
    {"op": "wait"},
    {"op": "set_setting", "key": "after", "value": "the wait"},
    {"op": "set_setting", "key": "never", "value": "run"},
]}), encoding="utf-8")
reader = subprocess.Popen([str(DEVTOOL), "store", "gone.json"], cwd=SCRATCH, env=ENVIRONMENT, stdin=subprocess.PIPE,
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
assert [json.loads(reader.stdout.readline())["op"] for _ in range(3)] == ["open", "set_setting", "wait"]
reader.stdout.close()
reader.stdin.close()
stderr = reader.stderr.read()
print("store:", reader.wait(timeout=600), stderr.strip())
assert reader.returncode == 1 and "panicked" not in stderr, stderr
assert re.fullmatch(r"golem-devtool: store: the standard output is closed: .+\n", stderr), stderr
answers, _ = steps([
    open_store("gone.db"),
    ({"op": "setting", "key": "before"}, "the wait"),
    ({"op": "setting", "key": "after"}, "the wait"),
    ({"op": "setting", "key": "never"}, None),
    CLOSE,
])

## A writer that fails

The process runs with a limit of 300 KiB on the size of the files it writes and with `SIGXFSZ`
ignored, so a write past the limit fails with `EFBIG` instead of ending the process, which SQLite
reports as a disk I/O error, and the writer fails on it as on any error of the file. It flushes two
accounts and then runs rounds of a setting and a transfer with a long note until the writer fails.
The calls after the failure, the flush and the close must answer that the store takes no more
changes, a second close must answer the same failure, the log must hold the line of the failure, and
the next open without the limit must find a sound file that holds a prefix of the rounds, as after a
kill. The store learns of the failure only once the writer has tried the failed batch change by
change, so the changes it took after the failing one answered ok and are lost: the cache holds more
rounds than the file, and the lost changes that `stats` counts are exactly the changes of those
rounds that the file lacks. That script closes the store in a step and exits with 0, since the
devtool closes at the end only a store that no step closed. A script that ends after the failure
without a close step exits with 1 and the failure of its last close on the standard error. A writer
thread that panics fails the store the same way: the step that makes it panic, a change, a flush,
a read through the writer such as a check, and both closes answer the one failure `the writer thread
panicked`, the change leaves the cache as it was, since a writer that is gone takes no change, and
the file holds every change before the panic.

In [ ]:
def limited():
    signal.signal(signal.SIGXFSZ, signal.SIG_IGN)
    resource.setrlimit(resource.RLIMIT_FSIZE, (size_limit_kib * 1024, size_limit_kib * 1024))


failure = r"the store takes no more changes, since the writer failed: .+ on the change \w+"
answers, stderr = steps([
    open_store("full.db"),
    ({"op": "join", "uuid": A, "name": "Alex", "at": 9000}, ANY),
    ({"op": "join", "uuid": B, "name": "Sam", "at": 9000}, ANY),
    ({"op": "create_account", "uuid": A, "balance": 1000000, "at": 9000, "note": "start"}, ANY),
    ({"op": "create_account", "uuid": B, "balance": 0, "at": 9000}, ANY),
    ({"op": "flush"}, None),
    ({"op": "repeat", "count": 100000, "steps": ROUND("counter", "round {i} " + "x" * 200)},
     Fails("failed", pattern=r"round \d+: " + failure)),
    ({"op": "flush"}, Fails("failed", pattern=failure)),
    ({"op": "stats"}, lambda stats: stats["failure"] and stats["lost"] > 0 and stats["pending"] == 0
     and stats["written"] + stats["lost"] == stats["queued"]),
    ({"op": "set_setting", "key": "after", "value": "the failure"}, Fails("failed", pattern=failure)),
    ({"op": "balance", "uuid": B}, ANY),
    ({"op": "setting", "key": "counter"}, ANY),
    ({"op": "close"}, Fails("failed", pattern=failure)),
    ({"op": "set_setting", "key": "after", "value": "the close"}, Fails("closed", "the store is closed")),
    ({"op": "close"}, Fails("failed", pattern=failure)),
], preexec=limited)
assert re.search(r"^\[golem\] store: the writer failed and the store takes no more changes: .+$", stderr, re.M), stderr
assert (SCRATCH / "full.db").stat().st_size <= size_limit_kib * 1024
stats, cached, cached_counter = answers[8], answers[10], int(answers[11])
answers, _ = steps([
    open_store("full.db"),
    ({"op": "check"}, lambda check: check["passed"]),
    ({"op": "setting", "key": "counter"}, ANY),
    ({"op": "balance", "uuid": B}, ANY),
    ({"op": "setting", "key": "after"}, None),
    CLOSE,
])
counter, rounds = int(answers[2] or 0), answers[3]
assert rounds in (counter, counter - 1), "the file holds %d transfers but the setting of round %d" % (rounds, counter)
assert rounds < cached, "the cache held %d rounds and the file as many, so the writer did not fail" % cached
missing = (cached - rounds) + (cached_counter - counter)
assert stats["lost"] == missing, "stats counts %d lost changes, the file lacks %d changes the store answered" % (
    stats["lost"], missing)
print("the writer failed at the limit of %d KiB: the file holds %d rounds and the setting of round %d, the cache held %d,"
      " so %d changes answered ok and were lost, as stats counts them" % (size_limit_kib, rounds, counter, cached, missing))
UNCLOSED = [
    {"op": "open", "path": "unclosed.db"},
    {"op": "join", "uuid": A, "name": "Alex", "at": 9000},
    {"op": "join", "uuid": B, "name": "Sam", "at": 9000},
    {"op": "create_account", "uuid": A, "balance": 1000000, "at": 9000, "note": "start"},
    {"op": "create_account", "uuid": B, "balance": 0, "at": 9000},
    {"op": "flush"},
    {"op": "repeat", "count": 100000, "steps": ROUND("counter", "round {i} " + "x" * 200)},
]
(SCRATCH / "unclosed.json").write_text(json.dumps({"steps": UNCLOSED}), encoding="utf-8")
unclosed = subprocess.run([str(DEVTOOL), "store", "unclosed.json"], cwd=SCRATCH, env=ENVIRONMENT, capture_output=True,
                          text=True, timeout=1200, preexec_fn=limited)
last_line = unclosed.stderr.splitlines()[-1]
print("a script that ends after the failure without a close exits %d: %s" % (unclosed.returncode, last_line))
assert unclosed.returncode == 1, (unclosed.returncode, unclosed.stderr)
assert re.fullmatch("golem-devtool: store: " + failure, last_line), unclosed.stderr
panicked = "the store takes no more changes, since the writer failed: the writer thread panicked"
answers, stderr = steps([
    open_store("panic.db"),
    ({"op": "set_setting", "key": "panic", "value": "before"}, None),
    ({"op": "flush"}, None),
    ({"op": "panic_writer"}, Fails("failed", panicked)),
    ({"op": "set_setting", "key": "panic", "value": "after"}, Fails("failed", panicked)),
    ({"op": "flush"}, Fails("failed", panicked)),
    ({"op": "check"}, Fails("failed", panicked)),
    ({"op": "durability"}, Fails("failed", panicked)),
    ({"op": "ledger", "limit": 5}, Fails("failed", panicked)),
    ({"op": "setting", "key": "panic"}, "before"),
    ({"op": "stats"}, lambda stats: stats["failure"] == "the writer thread panicked" and stats["queued"] == 1
     and stats["written"] == 1),
    ({"op": "close"}, Fails("failed", panicked)),
    ({"op": "close"}, Fails("failed", panicked)),
    open_store("panic.db"),
    ({"op": "setting", "key": "panic"}, "before"),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])
assert "[golem] store: the writer failed and the store takes no more changes: the writer thread panicked" in (
    stderr.splitlines()), stderr

## A batch that fails part way

A writer whose transaction fails applies its changes again one by one, so the file holds every
change before the one that fails, also those of the failed batch. The script lets the file of the
writer grow by 20 pages at most, holds the writer still for 500 ms and queues 200 settings of 3000
bytes in that time, so they reach the writer as one batch that needs far more pages. The batch
fails as on a full disk, which SQLite reports as a full database, and the changes then go in one
transaction each until one finds no page: the file must hold the first of them, each committed on
its own, and none from the one that failed on. A writer that dropped the whole failed batch would
leave none of them.

A second script changes the balance of an account in the file behind the cache. The transfer that
the cache then allows must fail in the file with `the file disagrees with the cache`, and `check`
of the next open must see the balance that its log does not sum to.

In [ ]:
PAUSE_MS = 500
full = r"the store takes no more changes, since the writer failed: the disk or the file is full: .+ on the change PutSetting"
answers, _ = steps([
    open_store("pages.db"),
    ({"op": "set_setting", "key": "start", "value": "x"}, None),
    ({"op": "flush"}, None),
    ({"op": "stats"}, keep("pages_before")),
    ({"op": "limit_pages", "pages": 20}, lambda limit: limit["max_page_count"] == limit["page_count"] + 20),
    ({"op": "pause_writer", "ms": PAUSE_MS}, None),
    ({"op": "repeat", "count": 200, "steps": [{"op": "set_setting", "key": "big{i}", "value": "x" * 3000}]},
     lambda burst: burst["calls"] * burst["nanos_per_call"] < PAUSE_MS * 1e6 / 2),
    ({"op": "flush"}, Fails("failed", pattern=full)),
    ({"op": "stats"}, keep("pages_after")),
    ({"op": "close"}, Fails("failed", pattern=full)),
    open_store("pages.db"),
    ({"op": "settings"}, keep("pages_file")),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])
before, after = SAVED["pages_before"], SAVED["pages_after"]
singles = after["written"] - before["written"]
assert 0 < singles < 200, "the writer committed %d changes of the failed batch one by one" % singles
assert after["batches"] - before["batches"] == singles, "not every change after the failed batch went alone: %s" % after
assert after["lost"] == 200 - singles and after["largest_batch"] == before["largest_batch"], after
expected = sorted(["start"] + ["big%d" % round for round in range(1, singles + 1)])
assert sorted(SAVED["pages_file"]) == expected, sorted(SAVED["pages_file"])
print("the batch of 200 settings failed at a limit of 20 pages, and the writer then committed the first %d of them "
      "one by one and lost %d" % (singles, after["lost"]))
behind = r"the store takes no more changes, since the writer failed: the file disagrees with the cache: the account " \
         r"of %s does not hold 500 cents on the change Transfer" % B
answers, _ = steps([
    open_store("behind.db"),
    ({"op": "join", "uuid": A, "name": "Alex", "at": 9000}, ANY),
    ({"op": "join", "uuid": B, "name": "Sam", "at": 9000}, ANY),
    ({"op": "create_account", "uuid": A, "balance": 1000, "at": 9000}, ANY),
    ({"op": "create_account", "uuid": B, "balance": 1000, "at": 9000}, ANY),
    ({"op": "execute_behind", "sql": "UPDATE accounts SET balance = 0 WHERE player = '%s'" % B}, None),
    ({"op": "transfer", "source": B, "target": A, "amount": 500, "at": 9100, "note": "pay"}, receipt(3, 500, 1500)),
    ({"op": "flush"}, Fails("failed", pattern=behind)),
    ({"op": "close"}, Fails("failed", pattern=behind)),
    open_store("behind.db"),
    ({"op": "balance", "uuid": B}, 0),
    ({"op": "check"}, lambda check: not check["passed"] and check["cache_differences"] == []
     and check["ledger_mismatches"] == [{"account": B, "balance": 0, "log": 1000}]),
    CLOSE,
])

## A store of the version before

A store that an older Golem made holds the schema of version 1, and the open must bring it to the
latest version with every row as it was. The cell builds such a file with the `sqlite3` of Python
from the first entry of the migrations in `schema.rs`, which must still be the text that the stores
of version 1 were made with, byte for byte, and fills every table. The open then migrates the file:
each row of version 1 must stand unchanged in the columns of version 1, each new column must hold
its default in every row, which for a player is what a first join gives them, the two new tables
must be empty, and the schema of the migrated file must be the schema of a new store. The store must
answer the rows as it answers its own, pass its check, and take every new call on them.

In [ ]:
SOURCE = (P.root / "crates" / "golem-store" / "src" / "schema.rs").read_text(encoding="utf-8")
SCHEMA_1 = re.search(r'const SCHEMA_1: &str = "\n(.*?)\n";\n', SOURCE, re.S).group(1)
assert hashlib.sha256(SCHEMA_1.encode("utf-8")).hexdigest() == (
    "6cce3d331863e94980c461e57294a35b501fd0fd612d3c45665d1e09b44da448"), "the schema of version 1 changed"
OLD_ALEX, OLD_SAM = "00000000-0000-4000-8000-0000000000b1", "00000000-0000-4000-8000-0000000000b2"
OVERWORLD = ("minecraft:overworld", 1.5, 64.0, -2.5, 90.0, 0.0)
NOWHERE = (None,) * 6
ROWS_1 = {
    "players": [(OLD_ALEX, "Old", 100, 200, 300, *OVERWORLD, *NOWHERE, 1, "Oldie"),
                (OLD_SAM, "Older", 10, 20, 30, *NOWHERE, "minecraft:the_nether", 8.0, 32.0, 8.0, 0.0, 45.0, 0, None)],
    "homes": [(OLD_ALEX, "base", *OVERWORLD)],
    "warps": [("shop", *OVERWORLD, None), ("vault", *OVERWORLD, "essentials.warps.vault")],
    "spawns": [("default", *OVERWORLD)],
    "mail": [(1, OLD_SAM, "Older", OLD_ALEX, "hello", 400, 0), (2, None, "Console", OLD_ALEX, "welcome", 500, 1)],
    "ignores": [(OLD_ALEX, OLD_SAM)],
    "accounts": [(OLD_ALEX, 700), (OLD_SAM, 300)],
    "transfers": [(1, 600, None, OLD_ALEX, 1000, "starting balance"), (2, 700, OLD_ALEX, OLD_SAM, 300, "pay")],
    "permission_groups": [("default", 0), ("admin", 10)],
    "permission_group_parents": [("admin", 0, "default")],
    "permission_group_nodes": [("default", 0, "essentials.home"), ("admin", 0, "essentials.*")],
    "permission_player_groups": [(OLD_ALEX, 0, "admin")],
    "permission_player_nodes": [(OLD_ALEX, 0, "-essentials.suicide")],
    "settings": [("motd", "Welcome back")],
}


def rows_of(connection, columns=None):
    # The rows of every table of version 1, in the columns of version 1 or in all.
    return {table: connection.execute("SELECT %s FROM %s ORDER BY rowid" % (
        ", ".join(columns[table]) if columns else "*", table)).fetchall() for table in ROWS_1}


def columns_of(connection):
    return {table: [row[1] for row in connection.execute("PRAGMA table_info(%s)" % table)] for table in ROWS_1}


def tables_of(connection):
    return [row[0] for row in connection.execute(
        "SELECT name FROM sqlite_schema WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]


OLD = SCRATCH / "old.db"
old = sqlite3.connect(OLD, isolation_level=None)
old.execute("PRAGMA journal_mode = WAL")
old.execute("PRAGMA foreign_keys = ON")
old.executescript("BEGIN;\n%s\nPRAGMA application_id = %d;\nPRAGMA user_version = 1;\nCOMMIT;" % (SCHEMA_1, 0x476F6C6D))
for table, rows in ROWS_1.items():
    old.executemany("INSERT INTO %s VALUES (%s)" % (table, ", ".join("?" * len(rows[0]))), rows)
COLUMNS_1 = columns_of(old)
SCHEMA_PLAYERS = next(table["columns"] for table in SCHEMA["tables"] if table["name"] == "players")
assert tables_of(old) == sorted(ROWS_1) and rows_of(old) == ROWS_1, "the file of version 1 is not what the cell wrote"
assert old.execute("PRAGMA foreign_key_check").fetchall() == []
assert old.execute("PRAGMA integrity_check").fetchall() == [("ok",)]
old.close()
shutil.copyfile(OLD, SCRATCH / "old-refused.db")
HERE = place(x=1.5, y=64.0, z=-2.5, yaw=90.0)
OLD_RECORD = {"uuid": OLD_ALEX, "name": "Old", "first_join": 100, "last_join": 200, "last_seen": 300, "logout": HERE,
              "back": None, "afk": True, "nickname": "Oldie", **TOGGLES}
OLD_MAIL = [mail(1, OLD_SAM, "Older", OLD_ALEX, "hello", 400, False),
            mail(2, None, "Console", OLD_ALEX, "welcome", 500, True)]
OLD_COUNTS = {"players": 2, "accounts": 2, "warps": 2, "settings": 1}
answers, _ = steps([
    ({"op": "open", "path": "old.db"}, OLD_COUNTS),
    ({"op": "player", "uuid": OLD_ALEX}, OLD_RECORD),
    ({"op": "player", "uuid": OLD_SAM}, lambda record: {name: record[name] for name in TOGGLES} == TOGGLES),
    ({"op": "past_names", "uuid": OLD_ALEX}, []),
    ({"op": "player_settings", "uuid": OLD_ALEX}, {}),
    ({"op": "mail", "receiver": OLD_ALEX}, OLD_MAIL),
    ({"op": "unread_mail", "receiver": OLD_ALEX, "at": LARGEST}, 1),
    ({"op": "warps"}, [{"name": "shop", "location": HERE, "permission": None, "last_owner": None},
                       {"name": "vault", "location": HERE, "permission": "essentials.warps.vault", "last_owner": None}]),
    ({"op": "balance", "uuid": OLD_ALEX}, 700),
    ({"op": "check"}, lambda check: check["passed"] and check["rows"]["player_names"] == 0
     and check["rows"]["player_settings"] == 0 and check["rows"]["players"] == 2),
    CLOSE,
])
new = sqlite3.connect("file:%s?immutable=1" % OLD, uri=True)
assert new.execute("PRAGMA user_version").fetchone()[0] == VERSION, "the open left the file at an older version"
assert rows_of(new, COLUMNS_1) == ROWS_1, "the migration changed a row of version 1"
DEFAULTS = {"players": (1, 0, 0, 1, None, None, 0, 0), "mail": (None, 0), "warps": (None,)}
for table, values in DEFAULTS.items():
    added = columns_of(new)[table][len(COLUMNS_1[table]):]
    held = set(new.execute("SELECT %s FROM %s" % (", ".join(added), table)).fetchall())
    assert held == {values}, "the new columns %s of %s hold %s" % (added, table, held)
    print("%-8s gained %s with %s in every row" % (table, ", ".join(added), values))
assert sorted(set(tables_of(new)) - set(ROWS_1)) == ["player_names", "player_settings"], tables_of(new)
new.close()
migrated = json.loads(devtool("store-describe", "old.db").stdout)
bare = lambda description: [dict(table, rows=0) for table in description["tables"]]
assert bare(migrated) == bare(SCHEMA) and migrated["indexes"] == SCHEMA["indexes"], "a migrated store differs from a new one"
answers, _ = steps([
    ({"op": "open", "path": "old.db"}, OLD_COUNTS),
    ({"op": "join", "uuid": OLD_ALEX, "name": "Newer", "at": 900}, {"first": False, "previous_name": "Old"}),
    ({"op": "past_names", "uuid": OLD_ALEX}, [{"name": "Old", "changed_at": 900}]),
    ({"op": "set_player_toggle", "uuid": OLD_ALEX, "toggle": "social_spy", "value": True}, None),
    ({"op": "set_player_setting", "uuid": OLD_ALEX, "key": "note", "value": "migrated"}, None),
    ({"op": "send_mail", "sender": OLD_ALEX, "sender_name": "Newer", "receiver": OLD_SAM, "text": "until later",
      "at": 900, "expires_at": 1000}, 3),
    ({"op": "set_warp", "name": "shop", "location": HERE, "permission": None, "owner": OLD_ALEX}, True),
    ({"op": "reopen"}, OLD_COUNTS),
    ({"op": "player", "uuid": OLD_ALEX}, dict(OLD_RECORD, name="Newer", last_join=900, last_seen=900, social_spy=True)),
    ({"op": "past_names", "uuid": OLD_ALEX}, [{"name": "Old", "changed_at": 900}]),
    ({"op": "player_settings", "uuid": OLD_ALEX}, {"note": "migrated"}),
    ({"op": "mail", "receiver": OLD_SAM}, [mail(3, OLD_ALEX, "Newer", OLD_SAM, "until later", 900, False, expires_at=1000)]),
    ({"op": "warp", "name": "shop"}, {"name": "shop", "location": HERE, "permission": None, "last_owner": OLD_ALEX}),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])
print("the open took a store of version 1 to version %d with every row of its %d tables as it was" % (VERSION, len(ROWS_1)))

## A migration that fails

A file that the migration cannot take must stay a store of the version before with every row as it
was. The one rule of version 2 that a row of version 1 can break is that of a legacy mail, so the
cell gives a copy of the file of version 1 a mail without a sender name, which no store of version 1
wrote and which version 2 takes for a legacy mail alone. The open must refuse the file before it
changes a line of it, and say what it found: the mail by its number, and with a second such mail
their count and the lowest of their numbers, and what to do. Once the mails have a sender name, a
table that stands in the way of the migration makes it fail inside its transaction, with the reason
of SQLite. Each time the file must still hold the version, the tables, the columns and the rows of
version 1, since a migration runs in one transaction. With the table gone, the open must migrate the
file.

In [ ]:
REFUSED = SCRATCH / "old-refused.db"
bad = sqlite3.connect(REFUSED, isolation_level=None)
bad.execute("INSERT INTO mail (sender_name, receiver, text, sent_at) VALUES ('', ?, 'a mail of no one', 600)", (OLD_ALEX,))
ROWS_REFUSED = rows_of(bad)
bad.close()
LEGACY_ALONE = (", and the schema version %d takes a mail without a sender name for a legacy mail alone, so give each "
                "such mail a sender name or remove it and open the store again" % VERSION)
no_migration = lambda how: [({"op": "open", "path": "old-refused.db"}, Fails(
    "open", "cannot open the store old-refused.db: the migration from the schema version 1 to %d %s" % (VERSION, how)))]


def kept_version_1(rows, beside=()):
    # The file as a refused or a failed migration must leave it, and the connection to change it.
    kept = sqlite3.connect(REFUSED, isolation_level=None)
    assert kept.execute("PRAGMA user_version").fetchone()[0] == 1, "the file left version 1 without a migration"
    assert tables_of(kept) == sorted([*ROWS_1, *beside]) and columns_of(kept) == COLUMNS_1, (
        "a migration that did not pass left a part of version %d" % VERSION)
    assert rows_of(kept) == rows, "a migration that did not pass changed a row"
    assert kept.execute("PRAGMA integrity_check").fetchall() == [("ok",)]
    return kept


answers, _ = steps(no_migration("is refused: the mail 3 has no sender name" + LEGACY_ALONE))
kept = kept_version_1(ROWS_REFUSED)
kept.execute("INSERT INTO mail (sender_name, receiver, text, sent_at) VALUES ('', ?, 'another', 700)", (OLD_SAM,))
ROWS_REFUSED = rows_of(kept)
kept.close()
answers, _ = steps(no_migration("is refused: 2 mails have no sender name, the lowest of them the mail 3" + LEGACY_ALONE))
kept = kept_version_1(ROWS_REFUSED)
kept.execute("UPDATE mail SET sender_name = 'Nobody' WHERE sender_name = ''")
kept.execute("CREATE TABLE player_settings (note TEXT)")
ROWS_REFUSED = rows_of(kept)
kept.close()
answers, _ = steps(no_migration("failed: table player_settings already exists"))
kept = kept_version_1(ROWS_REFUSED, beside=["player_settings"])
kept.execute("DROP TABLE player_settings")
kept.close()
answers, _ = steps([
    ({"op": "open", "path": "old-refused.db"}, OLD_COUNTS),
    ({"op": "mail", "receiver": OLD_ALEX}, [*OLD_MAIL, mail(3, None, "Nobody", OLD_ALEX, "a mail of no one", 600, False)]),
    ({"op": "mail", "receiver": OLD_SAM}, [mail(4, None, "Nobody", OLD_SAM, "another", 700, False)]),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])
assert json.loads(devtool("store-describe", "old-refused.db").stdout)["version"] == VERSION
print("the open refused a store of version 1 with mails that version %d refuses and one whose migration failed, left "
      "it at version 1 with every row each time, and migrated it once nothing stood in the way" % VERSION)

## A migration that is cut

A kill in the middle of a migration must leave the store of the version before or the store of the
new version, whole, and nothing between the two, since a migration is one transaction. The cell
fills a store of version 1 with 3000 players and 6000 mails, times one open that migrates a copy of
it, and then cuts six more such opens with `SIGKILL` at moments spread over that time. Behind each
cut the file must be sound, hold every row, and be either a store of version 1 with the tables and
columns of version 1 or one of the latest version with all of its tables and columns. The next open
must take it to the latest version and pass its check. Where a cut falls depends on the machine, so
the cell prints how many met each of the two states.

In [ ]:
import time

MANY, KILLS = 3000, 6
BIG = SCRATCH / "old-big.db"
crowd = ["0b160000-0000-4000-8000-%012x" % number for number in range(MANY)]
big = sqlite3.connect(BIG, isolation_level=None)
big.execute("PRAGMA journal_mode = WAL")
big.executescript("BEGIN;\n%s\nPRAGMA application_id = %d;\nPRAGMA user_version = 1;\nCOMMIT;" % (SCHEMA_1, 0x476F6C6D))
big.execute("BEGIN")
big.executemany("INSERT INTO players (uuid, name, first_join, last_join, last_seen) VALUES (?, ?, 1, 2, 3)",
                [(uuid, "crowd%d" % number) for number, uuid in enumerate(crowd)])
big.executemany("INSERT INTO mail (sender, sender_name, receiver, text, sent_at) VALUES (?, 'crowd', ?, 'hello there', 4)",
                [(crowd[number - 1], uuid) for number, uuid in enumerate(crowd)] * 2)
big.execute("COMMIT")
big.close()
CUT = SCRATCH / "cut.db"
(SCRATCH / "cut.json").write_text(json.dumps({"steps": [{"op": "open", "path": CUT.name}, {"op": "close"}]}),
                                  encoding="utf-8")


def migrating_open(kill_after=None):
    # Opens a copy of the store of version 1 in a process of its own, which migrates it, and answers how
    # long that took, or kills the process after a time and answers how it ended.
    shutil.copyfile(BIG, CUT)
    for beside in ("cut.db-wal", "cut.db-shm"):
        (SCRATCH / beside).unlink(missing_ok=True)
    started = time.monotonic()
    process = subprocess.Popen([str(DEVTOOL), "store", "cut.json"], cwd=SCRATCH, env=ENVIRONMENT,
                               stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if kill_after is None:
        assert process.wait(timeout=600) == 0, "the open that migrates the store failed"
        return time.monotonic() - started
    try:
        process.wait(timeout=kill_after)
    except subprocess.TimeoutExpired:
        os.kill(process.pid, signal.SIGKILL)
        process.wait(timeout=60)
    return process.returncode


whole = migrating_open()
found = {"version 1": 0, "version %d" % VERSION: 0}
killed = 0
for kill in range(1, KILLS + 1):
    status = migrating_open(whole * kill / (KILLS + 1))
    assert status in (0, -signal.SIGKILL), status
    killed += status != 0
    cut = sqlite3.connect(CUT, isolation_level=None)
    state = (cut.execute("PRAGMA user_version").fetchone()[0], len(columns_of(cut)["players"]), len(tables_of(cut)))
    rows = tuple(cut.execute("SELECT count(*) FROM %s" % table).fetchone()[0] for table in ("players", "mail"))
    sound = (cut.execute("PRAGMA integrity_check").fetchall(), cut.execute("PRAGMA foreign_key_check").fetchall())
    cut.close()
    assert state in ((1, len(COLUMNS_1["players"]), len(ROWS_1)), (VERSION, len(SCHEMA_PLAYERS), len(SCHEMA["tables"]))), (
        "a cut migration left version %d with %d columns of players and %d tables" % state)
    assert rows == (MANY, 2 * MANY) and sound == ([("ok",)], []), (rows, sound)
    found["version %d" % state[0]] += 1
    answers, _ = steps([
        ({"op": "open", "path": CUT.name}, {"players": MANY, "accounts": 0, "warps": 0, "settings": 0}),
        ({"op": "unread_mail", "receiver": crowd[0], "at": LARGEST}, 2),
        ({"op": "check"}, lambda check: check["passed"] and check["rows"]["mail"] == 2 * MANY),
        CLOSE,
    ])
    assert json.loads(devtool("store-describe", CUT.name).stdout)["version"] == VERSION
print("an open that migrates %d players and %d mails took %.0f ms with its process, and of %d opens that were cut at "
      "moments spread over that time %d were killed: %s, each a whole store that the next open took to version %d" % (
          MANY, 2 * MANY, whole * 1000, KILLS, killed,
          ", ".join("%d left %s" % (count, state) for state, count in found.items()), VERSION))

## Files that are no store

The open refuses a file that is not SQLite, a SQLite file of another application, one with tables
but no application id, a store of a newer schema version and one of a version below 0, each with
the message of `okf/store.md`, and leaves each file as it was, byte for byte. That holds also for a
database in the journal mode WAL that a kill left with its log `-wal`, one of another application
and one of a newer Golem, whose file and log both stay as they were, though a connection that may
write would checkpoint the log into the file and delete it. An empty file is a store of version 0,
which the open migrates to the latest version, and a file in a directory that does not exist is
refused.

In [ ]:
REFUSE = SCRATCH / "refuse"
REFUSE.mkdir()
(REFUSE / "text.db").write_text("these are the rules of the server\n", encoding="utf-8")
with sqlite3.connect(REFUSE / "other.db") as connection:
    connection.execute("PRAGMA application_id = 7")
    connection.execute("CREATE TABLE things (x)")
with sqlite3.connect(REFUSE / "tables.db") as connection:
    connection.execute("CREATE TABLE things (x)")
for name, version in (("newer.db", VERSION + 1), ("negative.db", -1)):
    answers, _ = steps([open_store("refuse/" + name), CLOSE])
    connection = sqlite3.connect(REFUSE / name)
    connection.execute("PRAGMA user_version = %d" % version)
    connection.close()
(REFUSE / "empty.db").write_bytes(b"")
KILLED_WAL = """
import os, signal, sqlite3, sys
connection = sqlite3.connect(sys.argv[1], isolation_level=None)
connection.execute("PRAGMA journal_mode = WAL")
connection.execute("PRAGMA wal_autocheckpoint = 0")
connection.execute("PRAGMA application_id = %d" % int(sys.argv[2]))
connection.execute("PRAGMA user_version = %d" % int(sys.argv[3]))
connection.execute("CREATE TABLE things (x)")
for row in range(200):
    connection.execute("INSERT INTO things VALUES (?)", ("row %d" % row,))
os.kill(os.getpid(), signal.SIGKILL)
"""
for name, application, version in (("other-wal.db", 7, 0), ("newer-wal.db", 0x476F6C6D, VERSION + 1)):
    killed = subprocess.run([sys.executable, "-I", "-c", KILLED_WAL, str(REFUSE / name), str(application), str(version)],
                            capture_output=True, text=True, timeout=60)
    assert killed.returncode == -signal.SIGKILL, killed
    assert (REFUSE / (name + "-wal")).stat().st_size > 0, "the kill left no log beside %s" % name
NEWER = "the store has the schema version %d, newer than the version %d this Golem knows" % (VERSION + 1, VERSION)
digest = lambda name: hashlib.sha256((REFUSE / name).read_bytes()).hexdigest() if (REFUSE / name).exists() else "gone"
FILES = {
    "text.db": "the file is not a SQLite database",
    "other.db": "the file is a SQLite database of another application, with the application id 0x7",
    "other-wal.db": "the file is a SQLite database of another application, with the application id 0x7",
    "tables.db": "the file is a SQLite database with 1 tables and indexes but no store of Golem",
    "newer.db": NEWER,
    "newer-wal.db": NEWER,
    "negative.db": "the store has the schema version -1, which no Golem writes",
}
KEPT = [*FILES, "other-wal.db-wal", "newer-wal.db-wal"]
before = {name: digest(name) for name in KEPT}
REFUSALS = [({"op": "open", "path": "refuse/" + name}, Fails("open", "cannot open the store refuse/%s: %s" % (name, why)))
            for name, why in FILES.items()]
answers, _ = steps(REFUSALS + [
    ({"op": "open", "path": "refuse/missing/store.db"},
     Fails("open", pattern=r"cannot open the store refuse/missing/store\.db: the file cannot be opened: .+")),
    open_store("refuse/empty.db"),
    CLOSE,
])
after = {name: digest(name) for name in KEPT}
assert before == after, "a refused file changed: %s" % [name for name in KEPT if before[name] != after[name]]
print("%d refused files and the logs of the two that a kill left in the journal mode WAL kept their bytes" % len(FILES))
described = json.loads(devtool("store-describe", "refuse/empty.db").stdout)
assert (described["version"], described["application_id"]) == (VERSION, 0x476F6C6D), described

## The cost of the calls of the server thread

`golem-devtool store-bench` fills a new store with 5000 players, each with three homes and an
account, every player but the first with two mails and an ignore of the player before them and every
tenth with a nickname, with 50 warps, three spawns, 20 settings, ten groups of 20 nodes beside the
group `default` with one node, and the first 100 players in a group and with a node of their own,
opens it again and times 2000 calls of each kind one by one. The devtool is a debug build. Every
kind must stay below the budget of a call on the server thread in its mean and in its 99th
percentile, and the writer behind the changes runs on its own thread, so its time does not count
against the server thread. The table of the page must name the same reads and changes, each inside
the budget in both builds as the page says, while the reload of the permissions stands in the table
as no call of the server thread and outside that claim. Each of the ten kinds of change changes the
store in every call, so the writer must write ten changes for every round.

The bench then runs five checks while a second thread reads a balance in a loop, as the server
thread would. A check holds the lock of the cache only to copy pointers to the players, the accounts
and the permissions, so every other call waits for it about as long as for a read, whatever the
number of players. The median time a check held the lock must stay below a tenth of the budget. The
longest read during the checks is printed beside the longest read of a quiet run of the same length,
since a single time on a shared machine holds the noise of its scheduler. The bench also times
reloads of the permissions as a resolver makes them once the generation moved, the set with its
generation, its TOML and the resolver that `golem-permissions` loads from it. A reload costs about
the whole budget of a call or more, so a port reloads off the server thread, and its mean must stay
below the bound that the page names.

The bench ends with a burst of three times `MAX_PENDING` calls of the ten changes in turn from one
thread, far faster than the writer commits, so changes find `MAX_PENDING` pending and wait for room.
The burst must wait, at most `MAX_PENDING` changes may be pending, the writer must commit full
transactions of `MAX_BATCH` changes, and the longest change, one that waited, must stay below the
bound that the page names, since a wait lasts until the writer committed the transaction it works
on.

In [ ]:
result = devtool("store-bench", "--path", "bench.db", "--players", bench_players, "--repeat", bench_repeat)
assert result.returncode == 0, result.stderr
BENCH = json.loads(result.stdout)
CHECK = BENCH["check"]
assert CHECK["passed"], BENCH
print("%-20s %9s %9s %9s %9s" % ("call", "mean us", "median us", "p99 us", "max us"))
for name, cost in BENCH["calls"].items():
    print("%-20s %9.2f %9.2f %9.2f %9.2f" % (name, cost["mean_us"], cost["median_us"], cost["p99_us"], cost["max_us"]))
over = [name for name, cost in BENCH["calls"].items() if cost["mean_us"] >= budget_us or cost["p99_us"] >= budget_us]
assert not over, "over the budget of %d us: %s" % (budget_us, over)
COST = PAGE.split("\n## The cost on the server thread\n", 1)[1].split("\n## ", 1)[0]
ROWS = [[cell.strip() for cell in line.split("|")[1:-1]] for line in COST.splitlines() if line.startswith("| `")]
TABLE = {re.sub(r"\((\d+)\)$", r"_\1", row[0].strip("`")): (row[1], [float(cell) for cell in row[2:6]])
         for row in ROWS}
CLAIM = re.search(r"Every read and every change of the table, the calls of the server thread, stays inside the budget "
                  r"of (\d+) ms in its mean and in its 99th percentile in both builds", " ".join(COST.split()))
assert CLAIM and int(CLAIM.group(1)) * 1000 == budget_us, "%s names no budget of %d us for its table" % (wiki, budget_us)
CALLED = {name for name, (kind, _) in TABLE.items() if kind in ("read", "change")}
assert CALLED == set(BENCH["calls"]), "the table of %s and the bench differ in %s" % (wiki, CALLED ^ set(BENCH["calls"]))
outside = [name for name in CALLED if max(TABLE[name][1]) >= budget_us]
assert not outside, "the table of %s puts %s over the budget it claims" % (wiki, outside)
assert [kind for kind, _ in TABLE.values() if kind not in ("read", "change")] == ["reload"], TABLE
writes = ["join", "set_back", "set_afk", "set_home", "send_mail", "mark_mail_read", "transfer", "deposit", "ignore",
          "set_setting"]
assert BENCH["writer"]["written"] == len(writes) * bench_repeat, "the writer wrote %d changes of %d calls" % (
    BENCH["writer"]["written"], len(writes) * bench_repeat)
server_ms = sum(BENCH["calls"][name]["mean_us"] * BENCH["calls"][name]["calls"] for name in writes) / 1000
print("open of %d players: %.1f ms, file %.1f MB, the process %s with the store open" % (
    bench_players, BENCH["open_ms"], BENCH["file_bytes"] / 1e6,
    "%.1f MB resident" % (BENCH["resident_kib_after_open"] / 1024) if BENCH["resident_kib_after_open"] else "of unknown size"))
print("the %d changes of the timed calls cost the server thread %.1f ms and the writer %.1f ms in %d transactions on "
      "its own thread" % (BENCH["writer"]["written"], server_ms, BENCH["writer"]["busy_ms"], BENCH["writer"]["batches"]))
print("%d checks of %.1f ms each held the lock of the cache %.1f us in the median and %.1f us at most, the longest read "
      "meanwhile took %.1f us and %.1f us in a quiet run" % (
          CHECK["checks"], CHECK["mean_ms"], CHECK["median_copy_us"], CHECK["longest_copy_us"],
          CHECK["longest_read_us"], CHECK["longest_read_quiet_us"]))
assert CHECK["median_copy_us"] < budget_us / 10, "a check holds the lock of the cache %.1f us" % CHECK["median_copy_us"]
RELOAD = BENCH["reload"]
reload_bound_ms = int(re.search(r"holds the mean reload of the bench store in the debug build below (\d+) ms",
                                " ".join(PAGE.split())).group(1))
print("%d reloads of the permissions took %.2f ms in the mean and %.2f ms in the 99th percentile, below the %d ms of %s"
      % (RELOAD["calls"], RELOAD["mean_us"] / 1000, RELOAD["p99_us"] / 1000, reload_bound_ms, wiki))
assert RELOAD["mean_us"] < reload_bound_ms * 1000, "a reload of the permissions takes %.2f ms" % (RELOAD["mean_us"] / 1000)
BURST = BENCH["burst"]
MAX_BATCH = int(re.search(r"\| `MAX_BATCH` \| (\d+),", PAGE).group(1))
burst_bound_ms = int(re.search(r"holds the longest change of the burst in the debug build below (\d+) ms",
                               " ".join(PAGE.split())).group(1))
longest_ms = BURST["calls"]["max_us"] / 1000
print("a burst of %d calls queued %d changes, %d of them waited for room with at most %d pending, the writer committed "
      "them in %d transactions of up to %d changes, %.1f ms each in the mean, and the longest change took %.1f ms or "
      "%.1f ticks of 50 ms, below the %d ms of %s" % (
          BURST["calls"]["calls"], BURST["changes"], BURST["waits"], BURST["largest_pending"], BURST["batches"],
          BURST["largest_batch"], BURST["busy_ms"] / BURST["batches"], longest_ms, longest_ms / 50, burst_bound_ms, wiki))
assert BURST["calls"]["calls"] == 3 * MAX_PENDING and BURST["waits"] > 0, BURST
assert BURST["largest_pending"] == MAX_PENDING, "%d changes were pending at once" % BURST["largest_pending"]
assert BURST["largest_batch"] == MAX_BATCH, "the largest transaction of the writer held %d changes" % BURST["largest_batch"]
assert longest_ms < burst_bound_ms, "a change of the burst took %.1f ms" % longest_ms

## Verdict

The cell clears the temporary directory. A run that reaches it passed.

In [ ]:
shutil.rmtree(SCRATCH)
slowest = max(BENCH["calls"].items(), key=lambda item: item[1]["mean_us"])
print("verdict: golem-store keeps the schema of %s, answered %d steps of %d scripts as expected with %d refusals, kept "
      "a prefix of its changes after a kill and after a failed writer, refused %d files that are no store, and its "
      "slowest call of the server thread, %s, took %.1f us in the mean" % (
          wiki, COUNTS["steps"], COUNTS["scripts"], COUNTS["refusals"], len(FILES) + 1, slowest[0],
          slowest[1]["mean_us"]))